# ICP-count regression - does the leading-var rise scale with connections?

**Decisive check #1 of the proxy decomposition** (harmonics workstream: `clients/ea/harmonics/consultation-2026/analysis/PROXY_DECOMPOSITION_20260806.md`). Dave's proposal, 6 Aug 2026; first run same evening; this notebook is the canonical version (supersedes `icp_regression_20260806.py`, kept for provenance). **Relocated to the IEEE paper directory 6 Aug pm (Dave's call)** - this regression is core evidence for the paper's organic-mechanism correction (the demand fleet is *adding* distributed capacitance, not merely drawing fewer lagging vars). Inputs are read in place from the harmonics resonance-screen panel and the power-factor contamination flags.

**Question.** The 2013-25 rise in overnight net leading vars at clean GXPs: does it scale with the number of connections (ICPs) behind each bus? The device-fleet story (EMI filter X-capacitors etc.) predicts a slope of roughly 100-250 VAr per connection with zero intercept. A GXP-level metering artefact has no mechanism to scale with connections - its prediction is the mean-only model.

**Pre-registered decision rule** (fixed before first computation, 6 Aug pm): SUPPORTED if the 95% CI on the slope excludes 0, the slope lies in [50, 400] VAr/ICP, and R^2 > 0.15. ARTEFACT-FAVOURED if the CI includes 0. Otherwise inconclusive. The rule is panel-agnostic and predates the 8-Aug panel rebase below.

**Data.** ICP counts: EMI `Retail/Datasets/MarketStructure/20260630_MarketShareTrendsByRootNSP.csv` (downloaded 6 Aug 2026; monthly ICP count by root NSP x retailer, Dec 2003 - Jun 2026; archived gzipped in `data_raw/`). Root NSP = POC + network + suffix; ICPs are summed per POC across networks and retailers, so embedded networks roll up to the grid POC. Metering endpoints: the pf half-hourly archive (`data/analysis/{year}_pf_by_gxp.parquet`). Clean cohort: power-factor `contamination_flag.csv`. Screen panel (`screen_by_gxp_year.csv`): retained for the original pre-registered run and as an independent cross-check of the archive-native medians.

**Two panels, reported separately, never blended.** (1) The **rebased panel** (8 Aug 2026): the archive-native 114-site clean endpoint cohort resolved into attribution units by the correspondence audit below - n = 96 units. This is the headline. (2) The **original pre-registered panel** (6 Aug 2026): the 90-site screen-table panel joined per-code to the registry - n = 83. Kept verbatim below as the labelled consistency result.

Assertion battery throughout = drift guard; run end-to-end after any data refresh.

> **Version note (8 Aug 2026, evening: panel rebase).** This notebook backs Section V of the paper
> *"From Lagging to Leading: The Measured Emergence of Standing Capacitance
> Behind Consumer Connections, 1997--2025"* -- V-A load-independence, V-B
> national sums + device estimate, V-C connection-scaling test and coefficient
> history. The paper ships two tiers with identical section numbering: a 14-pp
> journal version (OAJPE) and a 16-pp extended version.
>
> **The 8-Aug rebase** (Dave's question: "why only 83 of 114?") moved the
> scaling test from the legacy screen-table panel (n = 83 joined) onto the full
> archive-native 114-site cohort, via the correspondence audit below: 8
> retail-free industrial connections excluded by evidence, 2 sites without a
> registry endpoint excluded, 1 site unresolvable, and 14 cohort members
> aggregated into 7 attribution units (2 same-site pairs + 5 registry-churn
> groups discovered by the migration screen). Headline panel **n = 96 units
> (103 of the 114 cohort members)**; the original 83-panel run is kept verbatim
> below as the labelled consistency result. Sign conventions: screen cells
> leading-positive; paper-facing cells and the saved figure Q-signed
> (Q < 0 = leading), flagged in place. See `../../replication/README.md` for
> the panel taxonomy and figure map.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

HERE = Path.cwd()
ROOT = HERE.resolve().parents[4]  # gridlytics repo root
SCREEN = ROOT / "clients/ea/harmonics/consultation-2026/analysis/resonance-screen"
rng = np.random.default_rng(20260806)
PASSED = 0

def ok(cond, msg):
    global PASSED
    assert cond, f"ASSERTION FAILED: {msg}"
    PASSED += 1
    print(f"  ok: {msg}")

icp = pd.read_csv(HERE / "data_raw" / "icp_by_rootnsp_20260806.csv.gz")
ok(list(icp.columns) == ["Month ended", "Region", "Participant code", "Entity name",
                         "ICP count", "ICP share (%)"], "raw columns as expected")
ok(icp["Month ended"].nunique() == 271, "271 months Dec 2003 - Jun 2026")
icp = icp[icp["Region"].astype(str).str.len() == 13].copy()   # drop 226 malformed rows
icp["poc"] = icp["Region"].str[:7]
icp["month"] = pd.to_datetime(icp["Month ended"])
per_poc = icp.groupby(["poc", "month"], as_index=False)["ICP count"].sum()
chk = per_poc[per_poc.month == "2025-06-30"]["ICP count"]
ok(int(chk.sum()) == 2339766, "national ICP total at 2025-06 = 2,339,766")
ok(len(chk) == 152, "152 distinct POCs at 2025-06 (159 region strings roll up across networks)")
per_poc["year"] = per_poc.month.dt.year
yr = (per_poc[per_poc.year.isin([2013, 2025])]
      .groupby(["poc", "year"])["ICP count"].mean().unstack())
yr.columns = ["N2013", "N2025"]
yr["dN"] = yr.N2025 - yr.N2013
yr["Nbar"] = (yr.N2013 + yr.N2025) / 2
yr.round(0).to_csv(HERE / "icp_per_poc_20260806.csv")
print(f"per-POC table: {len(yr)} POCs with 2013 and 2025 counts")

  ok: raw columns as expected
  ok: 271 months Dec 2003 - Jun 2026


  ok: national ICP total at 2025-06 = 2,339,766
  ok: 152 distinct POCs at 2025-06 (159 region strings roll up across networks)
per-POC table: 179 POCs with 2013 and 2025 counts


## Panel rebase: correspondence audit and the 96-unit headline panel (8 Aug 2026)

Dave's 8-Aug question - *why does the scaling test use only 83 of the 114 cohort sites when the ICP data surely exists for all of them?* - turned out to have a deeper answer than a join fix. Joining the full archive-native cohort exposed that the **registry's root-NSP attribution and the metering's GXP boundaries are not the same partition of the network**, in three ways:

1. **Retail-free industrial connections** (8 sites). NZ Steel Glenbrook, Tiwai, the Kinleith mill feeders, Fonterra Lichfield, Brydone, Whirinaki: metered as GXPs, but their consumers are not in the retail registry (direct-purchase or single-industrial-ICP arrangements). The connection-scaling test is about the consumer device fleet; these sites have no retail fleet to scale with and are excluded *by identity, with the registry's own evidence* (transient 1-ICP entries naming RAYN / PANP / CTCT; POC-to-network mappings NZST / NZAS / VECT-at-Lichfield).
2. **Registry re-registrations** (2 sites excluded; 1 unresolvable). MainPower's Ashley POC has no registry series 2005-2015; Pacific Steel's Mangere 110 kV point enters the registry only in Dec 2015. And Linton is bound to a contaminated sibling by a 2023 transfer.
3. **Attribution churn between root NSPs** (5 groups) and **one-root two-point sites** (2 pairs). A migration screen over every panel member's monthly registry series (steps > max(300, 15%) classified physical vs registry-only by matching metered-P steps, seasonally differenced where confounded) shows the registry re-attributing blocks of 2k-27k ICPs between sibling POCs with **no physical load movement** - most dramatically Vector's North Shore trio (Albany 33/110 + Wairau Rd), where 20k+ ICPs swap seven times, including a 2013 "parking" of 27k at a substation metering ~0.2 MW. Within such a cluster only the **group** is a meaningful attribution unit, so the panel aggregates it (metering summed per half-hour before taking medians; registries summed).

The result: **103 of the 114 cohort members enter the test, resolved into 96 attribution units.** Physical network migrations (Wairau Rd 2014, NPL to Carrington St, Hinuera to the new ARI1102, Tauranga-area re-feeds, ...) need no correction - both registry and metering move together, and the M1/M2 specification handles growth. The screen's verified-physical and sub-threshold events are documented in the cell for the record.

Rebased cells are suffixed `_n` / `pan` and use a fresh seeded stream (20260806 is reserved for the byte-identical legacy run below). The pre-registered decision rule is applied unchanged.

In [2]:
# --- (r0) Correspondence audit: the 114 cohort resolved into attribution units (8 Aug 2026) ---
# Every exclusion and aggregation below is evidence-classed and asserted against the data.
ANA = ROOT / "clients/ea/power-factor/data/analysis"
META = ROOT / "clients/ea/power-factor/data/metadata"
fl = pd.read_csv(ROOT / "clients/ea/power-factor/replication/cache/contamination_flag.csv")
clean_codes = set(fl.loc[fl.contaminated == 0, "gxp"])
NIGHT, PEAK = list(range(1, 13)), list(range(35, 40))

def _endpoints(y):
    adf = pd.read_parquet(ANA / f"{y}_pf_by_gxp.parquet",
                          columns=["trading_date", "trading_period", "gxp_code", "P", "Q"])
    adf["lead"] = -adf.Q
    med = pd.concat([adf[adf.trading_period.isin(NIGHT)].groupby("gxp_code")["lead"].median().rename("night"),
                     adf[adf.trading_period.isin(PEAK)].groupby("gxp_code")["lead"].median().rename("peak")], axis=1)
    return adf, med
adf13, med13 = _endpoints(2013)
adf25, med25 = _endpoints(2025)
coh114 = sorted(g for g in med13.dropna().index.intersection(med25.dropna().index) if g in clean_codes)
ok(len(coh114) == 114, "archive-native clean endpoint cohort n = 114 (as in cell a2 below)")

nwk25 = dict(pd.read_csv(META / "2025_poc_nwk_mapping.csv").values)
nwk13 = dict(pd.read_csv(META / "2013_poc_nwk_mapping.csv").values)
raw = pd.read_csv(HERE / "data_raw" / "icp_by_rootnsp_20260806.csv.gz")  # pre-filter, for registry evidence
raw["month"] = pd.to_datetime(raw["Month ended"])
wide = per_poc.pivot(index="month", columns="poc", values="ICP count")

# Class A - retail-free direct-connect industrial / station offtakes (no retail registry at the
# endpoints; identity from the POC->network mapping and the registry's own transient rows).
EXCL_A = {
    "GLN0331": "NZ Steel Glenbrook (nwk NZST); never in the retail registry - the site's retail "
               "point is Counties' GLN0332 (contaminated, not in cohort)",
    "TWI2201": "NZAS Tiwai smelter (nwk NZAS); never in the retail registry",
    "KIN0111": "Kinleith mill 11 kV feeder; the mill's registry entry is KIN0112 with N = 1",
    "KIN0113": "Kinleith mill 11 kV feeder; as above (KIN0331, the Tokoroa network point, joins directly)",
    "LFD1101": "Fonterra Lichfield 110/11 kV (connection assets registered to VECT); never in the registry",
    "LFD1102": "Fonterra Lichfield, second transformer; as above",
    "BDE0111": "Brydone: ex-Rayonier Mataura MDF (nwk RAYN); registry = 1 ICP Dec 2003 - Jan 2005 only",
    "WHI0111": "Whirinaki: Contact station load 2013 (CTCT) -> Pan Pac mill 2025 (PANP); "
               "registry = 1 ICP Dec 2003 - Mar 2008 only",
}
ok(all(g in coh114 for g in EXCL_A), "all 8 class-A sites are cohort members")
ok(all(g not in yr.dropna().index for g in EXCL_A), "none of the 8 has a registry series at both endpoints")
ok(nwk25["GLN0331"] == "NZST" and nwk25["TWI2201"] == "NZAS" and nwk25["BDE0111"] == "RAYN"
   and nwk25["WHI0111"] == "PANP" and nwk13["WHI0111"] == "CTCT"
   and nwk25["LFD1101"] == "VECT" and nwk25["KIN0331"] == "POCO",
   "POC->network attributions confirm the industrial identities (2013 and 2025 mappings)")
bde = raw[raw["Region"].astype(str).str.startswith("BDE")]
whi = raw[raw["Region"].astype(str).str.startswith("WHI")]
ok(bde.month.max() <= pd.Timestamp("2005-01-31") and whi.month.max() <= pd.Timestamp("2008-03-31")
   and set(bde["ICP count"]) == {1} and set(whi["ICP count"]) == {1},
   "BDE/WHI registry evidence: single industrial ICP, series ending 2005-01 / 2008-03")
ok(not raw["Region"].astype(str).str.startswith(("LFD", "TWI")).any(),
   "LFD and TWI never appear in the retail registry at all")

# Class B - metered at both endpoints but no registry endpoint at the POC (re-registration history).
EXCL_B = {
    "ASY0111": "MainPower re-registration (nwk MPOW -> MPAS): registry absent 2005-2015, no 2013 N",
    "MNG1101": "Pacific Steel via Vector connection: registry begins 2015-12, N2025 = 2 (direct-connect scale)",
}
asy = wide["ASY0111"]
ok(asy.loc["2005-06":"2015-05"].isna().all() and pd.notna(asy.loc["2016-01-31"]),
   "ASY0111 registry gap 2005-2015 confirmed (consumers then registered under another root NSP)")
ok(wide["MNG1101"].first_valid_index() >= pd.Timestamp("2015-12-01"),
   "MNG1101 registry begins 2015-12 (metering exists from 2009: load was non-retail before)")

# Class E - unresolvable attribution: a material registry transfer binds the site to a
# non-clean sibling, so no clean attribution unit exists.
EXCL_E = {
    "LTN0331": "2023-05 transfer of 4,658 ICPs from BPE0331 (flagged contaminated); "
               "20% of LTN's N2025 - ungroupable, excluded",
}
d_ltn = wide["LTN0331"].diff().loc["2023-05-31"]
ok(abs(d_ltn - 4658) < 2 and int(fl.loc[fl.gxp == "BPE0331", "contaminated"].iloc[0]) == 1,
   "LTN0331 evidence: +4,658 step May 2023; counterpart BPE0331 is contaminated")

# Aggregation groups - the attribution unit is larger than the POC. Two kinds:
# (i) same-site pairs where the registry reports one root NSP for a two-point site;
# (ii) registry-churn groups discovered by the migration screen (monthly steps
#      |dN| > max(300, 15% of level) with no matching metered-P step at ~0.6 kW/ICP -
#      the registry re-attributes consumers between these POCs without physical change,
#      so only the group total is meaningful). Grouping is exact under both hypotheses
#      (physical or registry-only), so ambiguous cases group too.
AGG = {
    "KBY0661+2":   (["KBY0661", "KBY0662"], ["KBY0661"],
                    "Orion Kimberley: two clean 66 kV points, one registry root (KBY0661; series "
                    "starts 2013-07, so N2013 is a Jul-Dec mean; count stable ~970)"),
    "HTI0331+1101": (["HTI0331", "HTI1101"], ["HTI0331"],
                    "The Lines Company Hangatiki: 110 kV offtake HTI1101 (nwk LINE) commissioned "
                    "~2020 took ~5.9 MW off HTI0331 with no registry split - site sum restores "
                    "correspondence (HTI1101 is clean but post-2013, so not a cohort member)"),
    "ALB+WRD":     (["ALB0331", "ALB1101", "WRD0331"], ["ALB0331", "ALB1101", "WRD0331"],
                    "Vector North Shore churn trio: registry ping-pongs 20-27k ICPs (2013 parking at "
                    "then-unloaded WRD, 2019-09 and 2023-11 ~20k swaps with no P step); Wairau Rd's "
                    "2014 commissioning (physical, ~20 MW + 19.5k ICPs) is internal to the group"),
    "HEN+HEP":     (["HEN0331", "HEP0331"], ["HEN0331", "HEP0331"],
                    "Vector West churn pair: 2018-11 swap of 7.3k ICPs with no matching P transfer"),
    "PEN0221+0331": (["PEN0221", "PEN0331"], ["PEN0221", "PEN0331"],
                    "Vector Penrose churn pair: 2020-10 registry-only transfer of 4.6k; NOTE the 2014-10 "
                    "transfer of ~6k to contaminated PEN1101 leaks OUT of the group (~7% of group N)"),
    "CPK0111+0331": (["CPK0111", "CPK0331"], ["CPK0111", "CPK0331"],
                    "Wellington Central Park (CKHK): persistent +2.7k re-attributions 2014-2023 "
                    "between the pair; P evidence seasonal-confounded - grouped (safe either way)"),
    "HWB+SDN":     (["HWB0331", "SDN0331"], ["HWB0331", "SDN0331"],
                    "Aurora Dunedin (Halfway Bush + South Dunedin): 2018-12 transfer of 3.7k with "
                    "no P inflow at SDN (seasonal-differenced anomaly -0.7 MW vs +2.2 expected)"),
}
agg_members = [m for mem, _, _ in AGG.values() for m in mem]
ok(len(agg_members) == 15 and len(set(agg_members)) == 15, "7 groups, 15 member POCs, no overlap")
ok(sum(1 for m in agg_members if m in coh114) == 14 and "HTI1101" not in coh114,
   "14 of the 15 members are cohort members (HTI1101 rides in as clean post-2013 metering)")
ok(all(int(fl.loc[fl.gxp == m, "contaminated"].iloc[0]) == 0 for m in agg_members),
   "every group member is contamination-clean - groups never mix in flagged series")
ok(nwk25["HTI1101"] == "LINE", "HTI1101 maps to The Lines Company (network offtake, not industrial)")
d_wrd = wide["WRD0331"].diff()
ok(abs(d_wrd.loc["2013-06-30"] - 27052) < 2 and abs(d_wrd.loc["2013-09-30"] + 27029) < 2
   and abs(d_wrd.loc["2023-11-30"] + 21681) < 2,
   "North Shore churn evidence: the 2013 27k parking at WRD and the 2023-11 21.7k swap")
ok(abs(wide["HEP0331"].diff().loc["2018-11-30"] - 7348) < 2
   and abs(wide["SDN0331"].diff().loc["2018-12-31"] - 3659) < 2
   and abs(wide["PEN0221"].diff().loc["2020-10-31"] + 4630) < 2,
   "churn evidence: HEN->HEP 2018-11 (+7,348), HWB->SDN 2018-12 (+3,659), PEN0221 2020-10 (-4,630)")

# Physical migrations verified by matching P steps (self-consistent per-code, NO action):
# CST0331 <- NPL decommissioning 2019; KMO0331 <- TGA 2022 (+6 MW step); TMI/MTM re-feeds
# 2018-21 (P steps match); HIN0331 -> new ARI1102 2023 (-9.6 MW with -5,001 ICPs; ARI1102
# starts at 5,017); TGA0111 -> TGA0331 2018 (seasonal-differenced anomaly -1.8 ~ expected -1.7);
# HLY0331 <- Waipa siblings 2022-23 (anomalies +1.6/+2.1 ~ expected); CBG0111 -> HTU0331 2025-06;
# HWB0331 <- same-site HWB0332 consolidation 2020; BOB1101 <- BOB0331 2024; STK/WAI/INV/ISL/RFN
# splits (each side keeps its own registry+metering). Reversing or sub-threshold registry noise
# (HAM, MTN, GOR, HOR/KBY/TWZ seasonal oscillations) left per-code: endpoint calendar means are
# unaffected beyond ~5%/2,000-ICP materiality.
ari = wide["ARI1102"]
ok(ari.first_valid_index() == pd.Timestamp("2023-03-31") and abs(ari.iloc[ari.index.get_loc("2023-03-31")] - 5017) < 2,
   "HIN's counterpart: ARI1102 enters the registry 2023-03 at 5,017 (physical migration)")

drop = set(EXCL_A) | set(EXCL_B) | set(EXCL_E)
direct89 = [g for g in coh114 if g not in drop and g not in set(agg_members)]
ok(len(direct89) == 89, "89 cohort members join the registry directly, per-code")
ok(all(g in yr.dropna().index for g in direct89), "all 89 direct units have registry series at both endpoints")
n_units = len(direct89) + len(AGG)
print(f"cohort 114 -> {len(EXCL_A)} industrial + {len(EXCL_B)} no-endpoint + {len(EXCL_E)} unresolvable "
      f"excluded; {len(direct89)} direct + 14 members in {len(AGG)} groups = {n_units} attribution units "
      f"(103 of 114 members represented)")
ok(n_units == 96, "rebased panel: 96 attribution units")

  ok: archive-native clean endpoint cohort n = 114 (as in cell a2 below)


  ok: all 8 class-A sites are cohort members
  ok: none of the 8 has a registry series at both endpoints
  ok: POC->network attributions confirm the industrial identities (2013 and 2025 mappings)
  ok: BDE/WHI registry evidence: single industrial ICP, series ending 2005-01 / 2008-03


  ok: LFD and TWI never appear in the retail registry at all
  ok: ASY0111 registry gap 2005-2015 confirmed (consumers then registered under another root NSP)
  ok: MNG1101 registry begins 2015-12 (metering exists from 2009: load was non-retail before)
  ok: LTN0331 evidence: +4,658 step May 2023; counterpart BPE0331 is contaminated
  ok: 7 groups, 15 member POCs, no overlap
  ok: 14 of the 15 members are cohort members (HTI1101 rides in as clean post-2013 metering)
  ok: every group member is contamination-clean - groups never mix in flagged series
  ok: HTI1101 maps to The Lines Company (network offtake, not industrial)
  ok: North Shore churn evidence: the 2013 27k parking at WRD and the 2023-11 21.7k swap
  ok: churn evidence: HEN->HEP 2018-11 (+7,348), HWB->SDN 2018-12 (+3,659), PEN0221 2020-10 (-4,630)
  ok: HIN's counterpart: ARI1102 enters the registry 2023-03 at 5,017 (physical migration)
  ok: 89 cohort members join the registry directly, per-code
  ok: all 89 direct units ha

In [3]:
# --- (r1) Rebased panel build and the headline regression (n = 96 attribution units) ---
# Direct units use the per-GXP archive medians; aggregated units sum member Q per
# half-hour FIRST, then take the night median (medians are not additive).
rng8 = np.random.default_rng(20260808)  # rebase stream; legacy cells keep the 20260806 stream

site_of = {g: g for g in direct89}
for sid, (mem, _, _) in AGG.items():
    for m in mem:
        site_of[m] = sid

def site_night_med(adf, members):
    s = adf[adf.gxp_code.isin(members) & adf.trading_period.isin(NIGHT)]
    tot = s.groupby(["trading_date", "trading_period"])["lead"].sum()
    return float(tot.median())

rows = []
for g in direct89:
    rows.append((g, med13.loc[g, "night"], med25.loc[g, "night"], yr.loc[g, "N2013"], yr.loc[g, "N2025"]))
for sid, (mem, regs, _) in AGG.items():
    rows.append((sid, site_night_med(adf13, mem), site_night_med(adf25, mem),
                 yr.loc[regs, "N2013"].sum(), yr.loc[regs, "N2025"].sum()))
pan = pd.DataFrame(rows, columns=["site", "q13", "q25", "N2013", "N2025"]).set_index("site")
pan["dQ"] = pan.q25 - pan.q13
pan["Nbar"] = (pan.N2013 + pan.N2025) / 2
pan["dN"] = pan.N2025 - pan.N2013
npan = len(pan)
ok(npan == 96, "rebased panel n = 96 attribution units")

# Cross-check: on the old panel's sites still standing alone, the archive-native dQ
# must reproduce the screen table's dQ (independent computation paths). The screen table
# is frozen on the pre-14-Mar-2026 processed vintage (it also feeds the legacy panel,
# kept verbatim as the pre-registered run), so since the 13-Aug-2026 archive rebuild
# (pass-7 R3) the tie is tight in median with a ~0.06 MVAr tail at the POCs the old
# vintage carried as duplicate dual-network rows (PEN0331 et al.) or with fewer
# revision-covered half-hours (TWZ0331).
scr_r = pd.read_csv(SCREEN / "screen_by_gxp_year.csv")
piv_r = scr_r.pivot_table(index="gxp_code", columns="year", values="qc_night_med")
bal_r = piv_r.dropna(subset=[2013, 2025])
old83_r = [g for g in bal_r.index if g in clean_codes and g in yr.dropna().index]
overlap_r = [g for g in old83_r if g in pan.index]
agree = (pan.loc[overlap_r, "dQ"] - (bal_r.loc[overlap_r, 2025] - bal_r.loc[overlap_r, 2013])).abs()
print(f"screen-table agreement on {len(overlap_r)} standalone overlap sites: "
      f"median |diff| {agree.median():.4f} MVAr, max {agree.max():.4f}")
ok(len(overlap_r) == 75 and agree.median() < 0.01 and agree.max() < 0.10,
   "archive-native dQ ties to the screen table on all 75 overlap sites (median < 0.01 MVAr; "
   "max < 0.10 - the screen table embeds the pre-14-Mar-2026 vintage, see comment above)")

def ols(y, cols):
    A = np.column_stack([np.ones(len(y))] + cols)
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    sse = float(resid @ resid)
    sst = float(((y - y.mean()) ** 2).sum())
    return beta, 1 - sse / sst, len(y) * np.log(sse / len(y)) + 2 * A.shape[1], resid

V = 1e6  # MVAr/ICP -> VAr/ICP
Xn, XdNn, Yn = pan.Nbar.to_numpy(), pan.dN.to_numpy(), pan.dQ.to_numpy()
sse0 = float(((Yn - Yn.mean()) ** 2).sum())
aicn0 = npan * np.log(sse0 / npan) + 2
b1n, r2n, aicn1, residn = ols(Yn, [Xn])
pan["resid"] = residn
slope_n = b1n[1] * V
boots_n = np.array([ols(Yn[i], [Xn[i]])[0][1] for i in rng8.integers(0, npan, (4000, npan))]) * V
ci_n = np.percentile(boots_n, [2.5, 97.5])
ib_n = np.array([ols(Yn[i], [Xn[i]])[0][0] for i in rng8.integers(0, npan, (4000, npan))])
ici_n = np.percentile(ib_n, [2.5, 97.5])
ii, jj = np.triu_indices(npan, 1)
rat = (Yn[jj] - Yn[ii]) / (Xn[jj] - Xn[ii])
ts_n = float(np.median(rat[np.isfinite(rat)])) * V
origin_n = float((Xn @ Yn) / (Xn @ Xn)) * V
print(f"M1 (n={npan}): dQ = {b1n[0]:.2f} + {slope_n:.1f}e-6 * Nbar   R2={r2n:.3f}")
print(f"    bootstrap 95% CI [{ci_n[0]:.1f}, {ci_n[1]:.1f}] VAr/ICP | Theil-Sen {ts_n:.1f} | through-origin {origin_n:.1f}")
print(f"    intercept {b1n[0]:.2f} MVAr [95% CI {ici_n[0]:.2f}, {ici_n[1]:.2f}]")
print(f"    null (mean-only) AIC {aicn0:.1f} vs M1 AIC {aicn1:.1f}  (dAIC {aicn0-aicn1:.0f})")
ok(abs(slope_n - 229.6) < 1.0, "M1 slope 229.6 VAr/ICP (+/-1)")
ok(ici_n[0] < 0 < ici_n[1] and abs(b1n[0]) < 1.0,
   "intercept +0.3 MVAr, i.i.d. CI spans zero - and the r3 cluster cell finds it spans zero "
   "under every clustered scheme too; the device story's zero-intercept prediction is met")
ok(abs(r2n - 0.804) < 0.005, "R2 = 0.804 (+/-0.005)")
ok(ci_n[0] > 150 and ci_n[1] < 320, "bootstrap CI ~[199, 272]: excludes 0 by a distance")
ok(aicn0 - aicn1 > 130, "artefact null loses by dAIC > 130")
ok(abs(ts_n - 258.5) < 3 and abs(origin_n - 237.3) < 3,
   "robust estimators agree: Theil-Sen ~259, through-origin ~237")
ok(50 <= slope_n <= 400, "slope inside the pre-registered band [50, 400] - SUPPORTED under the 6-Aug rule")
if ci_n[1] > 400:
    print(f"NOTE: the CI upper tail ({ci_n[1]:.0f}) extends past the band ceiling 400; the rule tests the "
          f"point estimate, and the band was an order-of-magnitude prior, but report the tail honestly.")

  ok: rebased panel n = 96 attribution units
screen-table agreement on 75 standalone overlap sites: median |diff| 0.0030 MVAr, max 0.0560
  ok: archive-native dQ ties to the screen table on all 75 overlap sites (median < 0.01 MVAr; max < 0.10 - the screen table embeds the pre-14-Mar-2026 vintage, see comment above)


M1 (n=96): dQ = 0.29 + 229.6e-6 * Nbar   R2=0.804
    bootstrap 95% CI [198.6, 272.2] VAr/ICP | Theil-Sen 258.5 | through-origin 237.3
    intercept 0.29 MVAr [95% CI -0.21, 0.74]
    null (mean-only) AIC 303.5 vs M1 AIC 149.0  (dAIC 155)
  ok: M1 slope 229.6 VAr/ICP (+/-1)
  ok: intercept +0.3 MVAr, i.i.d. CI spans zero - and the r3 cluster cell finds it spans zero under every clustered scheme too; the device story's zero-intercept prediction is met
  ok: R2 = 0.804 (+/-0.005)
  ok: bootstrap CI ~[199, 272]: excludes 0 by a distance
  ok: artefact null loses by dAIC > 130
  ok: robust estimators agree: Theil-Sen ~259, through-origin ~237
  ok: slope inside the pre-registered band [50, 400] - SUPPORTED under the 6-Aug rule


In [4]:
# --- (r2) Rebased panel: growth decomposition M2 and influence checks ---
b2n, r2n_2, aicn2, _ = ols(Yn, [Xn, XdNn])
corr_n = float(np.corrcoef(Xn, XdNn)[0, 1])
m2b_boot = np.array([ols(Yn[i], [Xn[i], XdNn[i]])[0][2] for i in rng8.integers(0, npan, (4000, npan))]) * V
m2b_ci = np.percentile(m2b_boot, [2.5, 97.5])
print(f"M2: a={b2n[1]*V:.1f} VAr/ICP existing, b={b2n[2]*V:.1f} VAr per NEW ICP "
      f"[95% CI {m2b_ci[0]:.0f}, {m2b_ci[1]:.0f}], R2={r2n_2:.3f}, corr(Nbar,dN)={corr_n:.2f}")
ok(abs(b2n[2] * V - 111.4) < 2, "M2 new-connection term 111 VAr point estimate")
ok(m2b_ci[0] < 0 < m2b_ci[1], "M2 split UNIDENTIFIED on this panel: b's CI spans zero (collinearity 0.78)")
ok(abs(corr_n - 0.78) < 0.03, "Nbar-dN collinearity 0.78 - M2 is not interpretable site-by-site")
ok(abs(aicn2 - aicn1) < 2, "M2 and M1 are AIC-indistinguishable (|dAIC| < 2); with b unidentified, the single-slope model stands on parsimony")

loo_n = np.array([ols(np.delete(Yn, i), [np.delete(Xn, i)])[0][1] for i in range(npan)]) * V
imin, imax = int(np.argmin(loo_n)), int(np.argmax(loo_n))
print(f"leave-one-out slope range: [{loo_n.min():.1f}, {loo_n.max():.1f}] "
      f"(min = drop {pan.index[imin]}, max = drop {pan.index[imax]})")
ok(loo_n.min() > 210 and loo_n.max() < 260, "LOO slopes all within [220, 243] - no single unit drives the result")
ok(pan.index[imin] == "PEN0221+0331" and abs(loo_n[imin] - 220.3) < 1.5,
   "largest single influence: dropping the Penrose group -> 220 (-4%), verdict unchanged")
ok(loo_n.min() > 50 and loo_n.max() < 400, "every LOO slope stays inside the pre-registered band")

M2: a=208.7 VAr/ICP existing, b=111.4 VAr per NEW ICP [95% CI -233, 406], R2=0.808, corr(Nbar,dN)=0.78
  ok: M2 new-connection term 111 VAr point estimate
  ok: M2 split UNIDENTIFIED on this panel: b's CI spans zero (collinearity 0.78)
  ok: Nbar-dN collinearity 0.78 - M2 is not interpretable site-by-site
  ok: M2 and M1 are AIC-indistinguishable (|dAIC| < 2); with b unidentified, the single-slope model stands on parsimony
leave-one-out slope range: [220.3, 242.4] (min = drop PEN0221+0331, max = drop HEN+HEP)
  ok: LOO slopes all within [220, 243] - no single unit drives the result
  ok: largest single influence: dropping the Penrose group -> 220 (-4%), verdict unchanged
  ok: every LOO slope stays inside the pre-registered band


In [5]:
# --- (r3) Cluster-robust inference: the company as the resampling unit (10 Aug 2026) ---
# Peer-review finding (9 Aug panel): V-C's bootstrap CI, the leave-one-out sweep and the dAIC
# all treat the 96 units as independent, while Method 2 (Section IV-B) explicitly models
# company-level dependence ("sites under one company share a dose and are not independent")
# and the leverage sits in a few same-company Auckland groups. This cell redoes the inference
# with the COMPANY as the unit of resampling. Nothing above is altered; the i.i.d. numbers
# stand as published and are reprinted alongside for comparison.
from scipy import stats as _st

rngc = np.random.default_rng(20260810)  # cluster stream; earlier cells' draws are untouched
B_CL = 4000

# --- company attribution -----------------------------------------------------------------
unit_members = {g: [g] for g in direct89}
for _sid, (_mem, _regs, _) in AGG.items():
    unit_members[_sid] = list(_mem)
ok(set(unit_members) == set(pan.index), "every panel unit has a member list")
import sys as _sys
_sys.path.insert(0, str(META))
from edb_mapping import EDB_CODES as _EDB          # noqa: E402  (canonical 29-EDB table)

# The 2025 POC->nwk mapping is canonical WHERE IT NAMES AN EDB. It does not always: PEN0331
# maps to SHPK (Southpark), a code appearing NOWHERE in the ICP registry, while the registry's
# own root-NSP field puts all ~75k PEN0331 connections under VECT; TKU0331 maps to a generator
# code. Where the canonical mapping points outside the 29 EDBs we fall back to the registry's
# root NSP, majority-weighted by connections - the same source the counts themselves come from.
_reg = raw[raw["Region"].astype(str).str.len() == 13].copy()
_reg["poc"], _reg["nsp"] = _reg["Region"].str[:7], _reg["Region"].str[7:11]
_reg = _reg[_reg.month == _reg.month.max()]
_regmaj = (_reg.groupby(["poc", "nsp"])["ICP count"].sum().reset_index()
           .sort_values("ICP count").groupby("poc").nsp.last().to_dict())


def _owner(poc):
    c = nwk25.get(poc)
    return c if c in _EDB else _regmaj.get(poc, c)


for _p in sorted({p for u in pan.index for p in unit_members[u] if nwk25.get(p) not in _EDB}):
    print(f"  owner fallback {_p}: canonical mapping says {nwk25.get(_p)} (not an EDB) "
          f"-> registry root NSP {_owner(_p)}")

_comp, _straddle = {}, []
for u in pan.index:
    _w = {}
    for m in unit_members[u]:
        _w[_owner(m)] = _w.get(_owner(m), 0.0) + (float(yr.Nbar.get(m, 0.0)) if m in yr.index else 0.0)
    _comp[u] = max(_w, key=_w.get)
    if len(_w) > 1:
        _straddle.append((u, _comp[u], {k: round(v) for k, v in _w.items()}))
pan["company"] = [_comp[u] for u in pan.index]
for _u, _best, _w in _straddle:
    print(f"  straddling unit {_u}: connections by owner {_w} -> attributed to {_best}")
ok(set(pan.company) <= set(_EDB), "every panel unit attributed to one of the 29 canonical EDBs")
ok(_comp["PEN0221+0331"] == "VECT",
   "the Penrose group sits in Vector's cluster (registry root NSP, not the SHPK mapping row)")

comps = pan.company.to_numpy()
uniq = np.unique(comps)
G = len(uniq)
idx_of = {c: np.where(comps == c)[0] for c in uniq}
sz = pan.groupby("company").size().sort_values(ascending=False)
shareN = (pan.groupby("company").Nbar.sum() / pan.Nbar.sum()).sort_values(ascending=False)
print(f"cluster structure: {G} companies over {npan} units, mean {npan/G:.1f} units each; "
      f"most units {sz.index[0]} ({sz.iloc[0]}); most connections {shareN.index[0]} "
      f"({shareN.iloc[0]*100:.0f}%); {int((sz == 1).sum())} single-unit companies")
print("  top 5 by connections: " + ", ".join(f"{c} {shareN[c]*100:.0f}%" for c in shareN.index[:5]))
pan[["Nbar", "dQ", "dN", "company"]].to_csv(HERE / "panel_96_with_company.csv")

# --- analytic covariances: classical, heteroskedasticity-robust, cluster-robust ------------
A = np.column_stack([np.ones(npan), Xn])
XtXi = np.linalg.inv(A.T @ A)


def cr1(rv):
    """CR1 cluster-robust covariance of [intercept, slope], native units. Matches statsmodels."""
    meat = np.zeros((2, 2))
    for c in uniq:
        g = idx_of[c]
        s = A[g].T @ rv[g]
        meat += np.outer(s, s)
    return XtXi @ meat @ XtXi * (G / (G - 1)) * ((npan - 1) / (npan - 2))


s2 = float(residn @ residn) / (npan - 2)
Vcl = XtXi * s2                                             # classical
Vhc = XtXi @ (A.T @ np.diag(residn ** 2) @ A) @ XtXi * (npan / (npan - 2))   # HC1
Vcr = cr1(residn)                                           # CR1 cluster
tG = float(_st.t.ppf(0.975, G - 1))
tN = float(_st.t.ppf(0.975, npan - 2))

# --- pairs cluster bootstrap ---------------------------------------------------------------
cb_sl, cb_in = np.empty(B_CL), np.empty(B_CL)
for b in range(B_CL):
    i = np.concatenate([idx_of[uniq[k]] for k in rngc.integers(0, G, G)])
    bb, *_ = ols(Yn[i], [Xn[i]])
    cb_sl[b], cb_in[b] = bb[1] * V, bb[0]
cl_sl_ci = np.percentile(cb_sl, [2.5, 97.5])
cl_in_ci = np.percentile(cb_in, [2.5, 97.5])

# --- wild cluster bootstrap-t (Rademacher, null imposed) - the recommended test at small G --
pos = {c: k for k, c in enumerate(uniq)}
cmap = np.array([pos[c] for c in comps])


def wild_p(coef, restricted_fit, se_obs):
    """Two-sided wild cluster bootstrap-t p-value for one coefficient, null imposed."""
    t_o = coef / se_obs
    ts = np.empty(B_CL)
    for b in range(B_CL):
        w = rngc.choice(np.array([-1.0, 1.0]), size=G)[cmap]
        bs, _, _, rs = ols(restricted_fit[0] + restricted_fit[1] * w, [Xn])
        k = restricted_fit[2]
        ts[b] = bs[k] / float(np.sqrt(cr1(rs)[k, k]))
    return float((np.abs(ts) >= abs(t_o)).mean()), t_o


# slope null: dQ = mean + u   |   intercept null: dQ = b*Xn + u (through origin)
b_org = float((Xn @ Yn) / (Xn @ Xn))
p_sl, t_sl = wild_p(b1n[1], (Yn.mean(), Yn - Yn.mean(), 1), np.sqrt(Vcr[1, 1]))
p_in, t_in = wild_p(b1n[0], (b_org * Xn, Yn - b_org * Xn, 0), np.sqrt(Vcr[0, 0]))

print(f"\nSLOPE  point {slope_n:.1f} VAr/ICP")
print(f"  i.i.d. unit bootstrap   [{ci_n[0]:>6.1f}, {ci_n[1]:>6.1f}]   (as published, B=4000)")
print(f"  company-cluster boot    [{cl_sl_ci[0]:>6.1f}, {cl_sl_ci[1]:>6.1f}]   (B={B_CL}, G={G})")
print(f"  classical OLS           [{slope_n-tN*np.sqrt(Vcl[1,1])*V:>6.1f}, {slope_n+tN*np.sqrt(Vcl[1,1])*V:>6.1f}]"
      f"   SE {np.sqrt(Vcl[1,1])*V:.1f}")
print(f"  HC1 heteroskedastic     [{slope_n-tN*np.sqrt(Vhc[1,1])*V:>6.1f}, {slope_n+tN*np.sqrt(Vhc[1,1])*V:>6.1f}]"
      f"   SE {np.sqrt(Vhc[1,1])*V:.1f}")
print(f"  CR1 cluster t (df {G-1})  [{slope_n-tG*np.sqrt(Vcr[1,1])*V:>6.1f}, {slope_n+tG*np.sqrt(Vcr[1,1])*V:>6.1f}]"
      f"   SE {np.sqrt(Vcr[1,1])*V:.1f}")
print(f"  wild cluster bootstrap-t p = {p_sl:.4f}   -> the scaling law survives every scheme")

print(f"\nINTERCEPT  point {b1n[0]:.2f} MVAr per unit  ({b1n[0]*npan:.0f} MVAr over {npan} units, "
      f"{abs(b1n[0]*npan)/369*100:.0f}% of the 369 MVAr cohort deepening)")
for _nm, _lo, _hi in [
        ("i.i.d. unit bootstrap ", ici_n[0], ici_n[1]),
        ("company-cluster boot  ", cl_in_ci[0], cl_in_ci[1]),
        ("classical OLS         ", b1n[0]-tN*np.sqrt(Vcl[0, 0]), b1n[0]+tN*np.sqrt(Vcl[0, 0])),
        ("HC1 heteroskedastic   ", b1n[0]-tN*np.sqrt(Vhc[0, 0]), b1n[0]+tN*np.sqrt(Vhc[0, 0])),
        ("CR1 cluster t         ", b1n[0]-tG*np.sqrt(Vcr[0, 0]), b1n[0]+tG*np.sqrt(Vcr[0, 0]))]:
    print(f"  {_nm}  [{_lo:>6.2f}, {_hi:>6.2f}]  = [{_lo*npan:>5.0f}, {_hi*npan:>4.0f}] MVAr aggregate"
          f"   {'spans zero' if _lo < 0 < _hi else 'EXCLUDES zero'}")
print(f"  wild cluster bootstrap-t p = {p_in:.4f}  (H0: no connection-independent term)")

ok(cl_sl_ci[0] > 0 and p_sl < 0.01,
   "the connection-scaling slope survives company clustering (CI excludes zero, wild p < 0.01)")
ok(min(np.sqrt(Vcr[1, 1]), np.sqrt(Vcr[0, 0])) > 0, "cluster covariance is well formed")
print(f"  NOTE: CR1 SEs come out BELOW the heteroskedasticity-robust HC1 SEs "
      f"(slope {np.sqrt(Vcr[1,1])*V:.1f} vs {np.sqrt(Vhc[1,1])*V:.1f}); with G={G} and one "
      f"{sz.iloc[0]}-unit cluster this is the regime where CR estimators are least reliable, "
      f"so CR1 should not be used to TIGHTEN any claim. The wild bootstrap is the safer test.")

# --- leave-one-COMPANY-out ------------------------------------------------------------------
loco = {c: ols(Yn[comps != c], [Xn[comps != c]])[0][1] * V for c in uniq}
loco_s = pd.Series(loco).sort_values()
print(f"\nleave-one-company-out slope range [{loco_s.min():.1f}, {loco_s.max():.1f}] "
      f"(min = drop {loco_s.index[0]}, max = drop {loco_s.index[-1]}); "
      f"single-unit LOO was [{loo_n.min():.1f}, {loo_n.max():.1f}]")
print("  three largest deletions: " + ", ".join(
    f"{c} ({int(sz[c])} units, {shareN[c]*100:.0f}%) -> {loco[c]:.0f}" for c in shareN.index[:3]))
_out = {c: round(v) for c, v in loco.items() if not (50 <= v <= 400)}
print(f"  deletions leaving the pre-registered band [50, 400]: {_out or 'none'}")
ok(loco_s.min() > 200,
   "no single company's removal takes the slope below 200 VAr/ICP - not carried by one owner")

# --- scale-free reading: is R2 = 0.78 a size effect? ----------------------------------------
Yp = (Yn / Xn) * V
bsf, r2sf, _, _ = ols(Yp, [Xn])
print(f"\nscale-free variant: per-unit VAr/connection mean {Yp.mean():.0f}, median {np.median(Yp):.0f}, "
      f"IQR [{np.percentile(Yp,25):.0f}, {np.percentile(Yp,75):.0f}]")
print(f"  VAr/connection regressed on size: {bsf[1]*1e3:+.3f} VAr per 1000 connections, R2 = {r2sf:.3f}"
      f"  -> no material size dependence in the per-connection rate")
print(f"  estimator spread: OLS(extensive) {slope_n:.0f} | through-origin {origin_n:.0f} | "
      f"Theil-Sen {ts_n:.0f} | unweighted site mean {Yp.mean():.0f}")
ok(r2sf < 0.15, "per-connection rate shows no strong size dependence (scale-free R2 < 0.15)")

# --- what the dAIC does and does not license ------------------------------------------------
print(f"\ndAIC {aicn0-aicn1:.0f} is an i.i.d.-Gaussian quantity. With G={G} clusters averaging "
      f"{npan/G:.1f} units, the effective sample size is below {npan}; the wild cluster "
      f"bootstrap-t (p = {p_sl:.4f}) is the honest version of the same comparison and agrees.")


  ok: every panel unit has a member list


  owner fallback PEN0331: canonical mapping says SHPK (not an EDB) -> registry root NSP VECT
  owner fallback TKU0331: canonical mapping says GENE (not an EDB) -> registry root NSP LINE
  ok: every panel unit attributed to one of the 29 canonical EDBs
  ok: the Penrose group sits in Vector's cluster (registry root NSP, not the SHPK mapping row)
cluster structure: 25 companies over 96 units, mean 3.8 units each; most units POCO (23); most connections VECT (37%); 8 single-unit companies
  top 5 by connections: VECT 37%, POCO 17%, CKHK 11%, DUNE 5%, WAIK 5%



SLOPE  point 229.6 VAr/ICP
  i.i.d. unit bootstrap   [ 198.6,  272.2]   (as published, B=4000)
  company-cluster boot    [ 209.1,  313.3]   (B=4000, G=25)
  classical OLS           [ 206.4,  252.8]   SE 11.7
  HC1 heteroskedastic     [ 195.7,  263.6]   SE 17.1
  CR1 cluster t (df 24)  [ 203.6,  255.6]   SE 12.6
  wild cluster bootstrap-t p = 0.0000   -> the scaling law survives every scheme

INTERCEPT  point 0.29 MVAr per unit  (28 MVAr over 96 units, 8% of the 369 MVAr cohort deepening)
  i.i.d. unit bootstrap   [ -0.21,   0.74]  = [  -20,   71] MVAr aggregate   spans zero
  company-cluster boot    [ -0.52,   0.65]  = [  -49,   63] MVAr aggregate   spans zero
  classical OLS           [ -0.28,   0.86]  = [  -27,   83] MVAr aggregate   spans zero
  HC1 heteroskedastic     [ -0.18,   0.77]  = [  -18,   74] MVAr aggregate   spans zero
  CR1 cluster t           [ -0.16,   0.74]  = [  -15,   71] MVAr aggregate   spans zero
  wild cluster bootstrap-t p = 0.2910  (H0: no connection-independ

## Original pre-registered run (6 Aug 2026) - labelled consistency result

The cells below are the **original screen-table panel and regression, kept verbatim** (only this heading added). The screen table is fault-level-constrained (90 clean sites with both endpoints; 83 join the registry per-code), so it under-samples the large urban buses - in particular it contains none of the North Shore / West Auckland / Dunedin churn-group members above. Its per-code join also predates the correspondence audit. It remains exactly the run the decision rule was first applied to on 6 Aug, and its verdict (slope 253, CI 192-291, R^2 = 0.77, SUPPORTED) stands as the conservative consistency check on the rebased headline. Report both, never blend. `rng` here continues the original 20260806 stream from the load cell, so every number below reproduces the 6-Aug run exactly.

In [6]:
scr = pd.read_csv(SCREEN / "screen_by_gxp_year.csv")
fl = pd.read_csv(ROOT / "clients/ea/power-factor/replication/cache/contamination_flag.csv")
piv = scr.pivot_table(index="gxp_code", columns="year", values="qc_night_med")
bal = piv.dropna(subset=[2013, 2025]).copy()
bal["dQ"] = bal[2025] - bal[2013]
clean_codes = set(fl.loc[fl.contaminated == 0, "gxp"])
d_all = bal.join(yr, how="inner")
d = d_all[d_all.index.isin(clean_codes)].dropna(subset=["dQ", "Nbar", "dN"]).copy()
X, XdN, Y = d.Nbar.to_numpy(), d.dN.to_numpy(), d.dQ.to_numpy()
n = len(d)
ok(n == 83, "clean joined cohort n = 83")
ok(len(d_all) == 113, "all-flag joined cohort n = 113")

  ok: clean joined cohort n = 83
  ok: all-flag joined cohort n = 113


In [7]:
def ols(y, cols):
    A = np.column_stack([np.ones(len(y))] + cols)
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    sse = float(resid @ resid)
    sst = float(((y - y.mean()) ** 2).sum())
    return beta, 1 - sse / sst, len(y) * np.log(sse / len(y)) + 2 * A.shape[1], resid

V = 1e6  # MVAr/ICP -> VAr/ICP
sse0 = float(((Y - Y.mean()) ** 2).sum())
aic0 = n * np.log(sse0 / n) + 2
b1, r2, aic1, resid1 = ols(Y, [X])
slope = b1[1] * V
boots = np.array([ols(Y[i], [X[i]])[0][1] for i in (rng.integers(0, n, (4000, n)))]) * V
ci = np.percentile(boots, [2.5, 97.5])
ii, jj = np.triu_indices(n, 1)
ts = float(np.median(((Y[jj] - Y[ii]) / (X[jj] - X[ii]))[np.isfinite((Y[jj]-Y[ii])/(X[jj]-X[ii]))])) * V
origin = float((X @ Y) / (X @ X)) * V
print(f"M1: dQ = {b1[0]:.2f} + {slope:.1f}e-6 * Nbar   R2={r2:.3f}")
print(f"    bootstrap 95% CI [{ci[0]:.1f}, {ci[1]:.1f}] VAr/ICP | Theil-Sen {ts:.1f} | through-origin {origin:.1f}")
print(f"    null (mean-only) AIC {aic0:.1f} vs M1 AIC {aic1:.1f}  (dAIC {aic0-aic1:.0f})")
ok(abs(slope - 252.8) < 1.0, "M1 slope 252.8 VAr/ICP (+/-1)")
ok(abs(b1[0]) < 0.2, "intercept small (<0.2 MVAr; scheme-dependence of the intercept is examined on the rebased panel, cell r3)")
ok(abs(r2 - 0.769) < 0.005, "R2 = 0.769 (+/-0.005)")
ok(ci[0] > 150 and abs(ci[0] - 191.8) < 5, "CI lower ~192, excludes 0 by a distance")
ok(abs(ci[1] - 291.4) < 5, "CI upper ~291")
ok(aic0 - aic1 > 100, "artefact null loses by dAIC > 100")
ok(abs(ts - 263.5) < 3, "Theil-Sen agrees (~264)")
ok(abs(origin - 251.0) < 3, "through-origin agrees (~251)")
ok(50 <= slope <= 400, "slope inside pre-registered band [50, 400]")

M1: dQ = -0.05 + 252.8e-6 * Nbar   R2=0.769
    bootstrap 95% CI [191.8, 291.4] VAr/ICP | Theil-Sen 263.5 | through-origin 251.0
    null (mean-only) AIC 222.1 vs M1 AIC 102.5  (dAIC 120)
  ok: M1 slope 252.8 VAr/ICP (+/-1)
  ok: intercept small (<0.2 MVAr; scheme-dependence of the intercept is examined on the rebased panel, cell r3)
  ok: R2 = 0.769 (+/-0.005)
  ok: CI lower ~192, excludes 0 by a distance
  ok: CI upper ~291
  ok: artefact null loses by dAIC > 100
  ok: Theil-Sen agrees (~264)
  ok: through-origin agrees (~251)
  ok: slope inside pre-registered band [50, 400]


In [8]:
b2, r2_2, aic2, _ = ols(Y, [X, XdN])
corr = float(np.corrcoef(X, XdN)[0, 1])
print(f"M2: a={b2[1]*V:.1f} VAr/ICP existing, b={b2[2]*V:.1f} VAr per NEW ICP, R2={r2_2:.3f}, corr(Nbar,dN)={corr:.2f}")
ok(abs(b2[2] * V) < 60, "new-connection term small (<60 VAr) - accumulation dominates")
ok(abs(corr - 0.72) < 0.03, "Nbar-dN collinearity 0.72 (interpret M2 cautiously)")
ok(aic2 > aic1, "M2 does not beat M1 on AIC")

loo = np.array([ols(np.delete(Y, i), [np.delete(X, i)])[0][1] for i in range(n)]) * V
print(f"leave-one-out slope range: [{loo.min():.1f}, {loo.max():.1f}]"
      f" (min = drop {d.index[np.argmin(loo)]}, max = drop {d.index[np.argmax(loo)]})")
ok(loo.min() > 230 and loo.max() < 280, "LOO slopes all within [230, 280] - no single bus drives the result")
ipen = list(d.index).index("PEN0331")
ok(abs(loo[ipen] - 239.1) < 1, "dropping high-leverage PEN0331 -> 239 (-5% only)")

M2: a=245.7 VAr/ICP existing, b=37.8 VAr per NEW ICP, R2=0.769, corr(Nbar,dN)=0.72
  ok: new-connection term small (<60 VAr) - accumulation dominates
  ok: Nbar-dN collinearity 0.72 (interpret M2 cautiously)
  ok: M2 does not beat M1 on AIC
leave-one-out slope range: [239.1, 269.6] (min = drop PEN0331, max = drop TAK0331)
  ok: LOO slopes all within [230, 280] - no single bus drives the result
  ok: dropping high-leverage PEN0331 -> 239 (-5% only)


In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

d["resid"] = resid1
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(X / 1000, Y, s=22, c="0.15", alpha=0.75, lw=0, zorder=3)
xx = np.linspace(0, X.max() * 1.04, 50)
ax.plot(xx / 1000, b1[0] + b1[1] * xx, c="0.45", lw=1.4, zorder=2)
for code in d.reindex(d.resid.abs().sort_values(ascending=False).index).head(4).index.tolist() + ["PEN0331"]:
    r = d.loc[code]
    ax.annotate(code, (r.Nbar / 1000, r.dQ), textcoords="offset points",
                xytext=(5, 4), fontsize=7, color="0.35")
ax.set_xlabel("Mean ICP count behind GXP, 2013-25 (thousands)")
ax.set_ylabel("Rise in night-median net leading MVAr, 2013-25")
ax.set_title(f"Clean cohort (n={n}): slope {slope:.0f} VAr/ICP "
             f"[95% CI {ci[0]:.0f}-{ci[1]:.0f}], $R^2$={r2:.2f}, intercept~0", fontsize=10)
ax.grid(True, lw=0.3, alpha=0.5)
fig.tight_layout()
fig.savefig(HERE / "icp_regression_scatter_20260806.png", dpi=150)
fig.savefig(HERE / "icp_regression_scatter_20260806.pdf")
print("figure saved (working style - restyle before any paper use)")
plt.show()

figure saved (working style - restyle before any paper use)


## National sums, the bottom-up device estimate, and consistency (added 7 Aug 2026)

The IEEE paper (Section V-B) quotes the size of the organic rise and compares it with a bottom-up order-of-magnitude estimate of the device-fleet standing capacitance ("Fermi estimate" in the working docs). The cells below make those numbers notebook-asserted. The national sums need no ICP join and are **independent of the 8-Aug panel rebase**: cell (a) runs on the legacy screen-table endpoint panel (n = 90) that the harmonics documents quote; cell (a2) runs on the full archive-native 114-site cohort - these are the numbers the PAPER quotes in V-B, and they are unchanged by the rebase (the rebase re-partitions the cohort into attribution units for the *regression*; the cohort sums are the same 114 sites either way). Sign convention in these cells stays leading-positive (screen convention); the paper flips to its own Q < 0 = leading. The figure cell below regenerates the paper's two-panel figure from the rebased panel.

In [10]:
# (a) Clean-panel national sums: every clean bus with the night-median defined in both
# 2013 and 2025 - no ICP join required, so n = 90 (the 83 above are these minus join losses).
cb = bal[bal.index.isin(clean_codes)]
s13, s25 = float(cb[2013].sum()), float(cb[2025].sum())
swing = s25 - s13
dq90 = cb[2025] - cb[2013]
risers, fallers = float(dq90[dq90 > 0].sum()), float(dq90[dq90 < 0].sum())
pivp = scr.pivot_table(index="gxp_code", columns="year", values="qc_p99")
cbp = pivp.dropna(subset=[2013, 2025])
cbp = cbp[cbp.index.isin(clean_codes)]
swing_p99 = float(cbp[2025].sum() - cbp[2013].sum())
HH_M = 2.0  # NZ households, millions (Stats NZ order; the device estimate's denominator)
per_hh = sorted([swing_p99 / HH_M, swing / HH_M, risers / HH_M])  # MVAr per M-hh = VAr/hh
print(f"clean endpoint panel n={len(cb)}: night-median {s13:+.1f} (2013) -> {s25:+.1f} MVAr (2025), swing {swing:+.1f}")
print(f"  risers {risers:+.1f} / fallers {fallers:+.1f}; p99-estimator swing {swing_p99:+.1f} (n={len(cbp)})")
print(f"  per household ({HH_M:.1f}M hh): {per_hh[0]:.0f}-{per_hh[-1]:.0f} VAr/hh across estimators")
ok(len(cb) == 90 and len(cbp) == 90, "clean endpoint panel n = 90 (both estimators)")
ok(abs(s13 + 65.7) < 1 and abs(s25 - 226.2) < 1 and abs(swing - 291.8) < 1,
   "night-median sums: -66 MVAr (2013) -> +226 MVAr (2025), swing +292")
ok(abs(risers - 322.7) < 1 and abs(fallers + 30.9) < 1 and abs(swing_p99 - 226.9) < 1,
   "risers +323 / fallers -31; p99-estimator swing +227")
ok(112 < per_hh[0] < 115 and 159 < per_hh[-1] < 163,
   "per-household equivalents span 113-161 VAr/hh across estimators")

clean endpoint panel n=90: night-median -65.7 (2013) -> +226.2 MVAr (2025), swing +291.8
  risers +322.7 / fallers -30.9; p99-estimator swing +226.9 (n=90)
  per household (2.0M hh): 113-161 VAr/hh across estimators
  ok: clean endpoint panel n = 90 (both estimators)
  ok: night-median sums: -66 MVAr (2013) -> +226 MVAr (2025), swing +292
  ok: risers +323 / fallers -31; p99-estimator swing +227
  ok: per-household equivalents span 113-161 VAr/hh across estimators


In [11]:
# (a2) Full clean endpoint cohort, archive-native (D1, Dave 7 Aug eve). The screen table
# above is fault-level-constrained (90 sites); the pf half-hourly archive itself defines
# the full clean cohort with both endpoints metered in BOTH windows: n = 114 - the same
# cohort as the parallel-shift (unmasking) test, whose headline numbers must reproduce
# here as a cross-validation (they are the paper's V-A load-independence figures, until
# now asserted only in the harmonics workings notebook - native from this cell on).
ANA = ROOT / "clients/ea/power-factor/data/analysis"
NIGHT, PEAK = range(1, 13), range(35, 40)
arows = {}
for y in (2013, 2025):
    adf = pd.read_parquet(ANA / f"{y}_pf_by_gxp.parquet",
                          columns=["trading_period", "gxp_code", "P", "Q"])
    adf["lead"] = -adf.Q
    arows[y] = pd.concat([
        adf[adf.trading_period.isin(NIGHT)].groupby("gxp_code")["lead"].median().rename("night"),
        adf[adf.trading_period.isin(PEAK)].groupby("gxp_code")["lead"].median().rename("peak"),
        adf[adf.trading_period.isin(PEAK)].groupby("gxp_code")["P"].median().rename("peak_P"),
        adf.groupby("gxp_code")["lead"].quantile(0.99).rename("p99")], axis=1)
a13, a25 = arows[2013], arows[2025]
cohort114 = [g for g in a13.dropna().index.intersection(a25.dropna().index) if g in clean_codes]
n114 = len(cohort114)
s13_114 = float(a13.loc[cohort114, "night"].sum())
s25_114 = float(a25.loc[cohort114, "night"].sum())
swing114 = s25_114 - s13_114
dq114 = a25.loc[cohort114, "night"] - a13.loc[cohort114, "night"]
risers114, fallers114 = float(dq114[dq114 > 0].sum()), float(dq114[dq114 < 0].sum())
swing114_p99 = float(a25.loc[cohort114, "p99"].sum() - a13.loc[cohort114, "p99"].sum())
per_hh114 = sorted([swing114_p99 / HH_M, swing114 / HH_M, risers114 / HH_M])
dk114 = a25.loc[cohort114, "peak"] - a13.loc[cohort114, "peak"]
ratio114 = float((dk114 / dq114.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).median())
pload114 = float((a25.loc[cohort114, "peak_P"] / a13.loc[cohort114, "peak_P"] - 1).median()) * 100
print(f"archive-native clean cohort n={n114}: night-median {s13_114:+.1f} (2013) -> {s25_114:+.1f} MVAr (2025), deepening {swing114:+.1f}")
print(f"  risers {risers114:+.1f} / fallers {fallers114:+.1f}; p99-estimator deepening {swing114_p99:+.1f}")
print(f"  per household ({HH_M:.1f}M hh): {per_hh114[0]:.0f}-{per_hh114[-1]:.0f} VAr/hh across estimators")
print(f"  parallel-shift cross-check: night rise med {dq114.median():+.2f}, peak rise med {dk114.median():+.2f}, "
      f"ratio med {ratio114:.2f}, peak load change {pload114:+.1f}%")
ok(n114 == 114, "archive-native clean endpoint cohort n = 114 (night + peak medians, both years)")
ok(abs(dq114.median() - 2.25) < 0.03 and abs(dk114.median() - 2.42) < 0.03 and abs(ratio114 - 1.06) < 0.03
   and abs(pload114 - 2.9) < 0.3,
   "parallel-shift test reproduces natively: +2.25 night / +2.42 peak / ratio 1.06 / peak load +2.9%")
ok(abs(s13_114 + 187.5) < 1 and abs(s25_114 - 181.5) < 1 and abs(swing114 - 369.0) < 1,
   "114-site night-median sums: -188 (2013) -> +182 MVAr (2025), deepening +369")
ok(abs(risers114 - 427.4) < 1 and abs(fallers114 + 58.4) < 1 and abs(swing114_p99 - 313.7) < 1.5,
   "risers +427 / fallers -58; p99-estimator deepening +314")
ok(155 < per_hh114[0] < 159 and 212 < per_hh114[-1] < 216,
   "per-household 157-214 VAr/hh across estimators (2.0M hh)")
ok(150 < swing114_p99 and swing114 < 600,
   "ORDER CHECK ONLY -- 114-site deepening (+314 to +369 MVAr) is the same ORDER as the 150-600 "
   "MVAr bottom-up band, and per-household (157-214) the same order as the 100-200 central band. "
   "The deepening is a 2013-25 CHANGE; the band is a standing LEVEL. Do NOT report the change as "
   "lying inside the band -- see the 'consistency' WARNING (external review 2, 24 Aug 2026)")

archive-native clean cohort n=114: night-median -187.6 (2013) -> +181.5 MVAr (2025), deepening +369.2
  risers +427.5 / fallers -58.4; p99-estimator deepening +313.5
  per household (2.0M hh): 157-214 VAr/hh across estimators
  parallel-shift cross-check: night rise med +2.25, peak rise med +2.42, ratio med 1.06, peak load change +2.9%
  ok: archive-native clean endpoint cohort n = 114 (night + peak medians, both years)
  ok: parallel-shift test reproduces natively: +2.25 night / +2.42 peak / ratio 1.06 / peak load +2.9%
  ok: 114-site night-median sums: -188 (2013) -> +182 MVAr (2025), deepening +369
  ok: risers +427 / fallers -58; p99-estimator deepening +314
  ok: per-household 157-214 VAr/hh across estimators (2.0M hh)
  ok: ORDER CHECK ONLY -- 114-site deepening (+314 to +369 MVAr) is the same ORDER as the 150-600 MVAr bottom-up band, and per-household (157-214) the same order as the 100-200 central band. The deepening is a 2013-25 CHANGE; the band is a standing LEVEL. Do NOT rep

In [12]:
# (a3) The coefficient's history (Dave, 7 Aug: "can we look at how this has changed in
# time"; REBASED 8 Aug onto the 96-unit panel). Year-by-year cross-sectional slope beta(t)
# of night-median leading LEVEL on ICP count, same units every year: panel units with
# metering AND registry present throughout 2009-2025 (n = 92; drops BPD1101, CUL0661,
# PAO1101 for metering gaps and the Kimberley group, whose registry starts 2013).
# Aggregated units sum member Q per half-hour and sum member registries; members that do
# not yet exist contribute nothing (physically correct: the substation wasn't there).
# Levels, so each year's slope is the whole connection-scaling class; its MOVEMENT is the
# accumulation. The archive starts 2009 - four pre-window years of baseline.
rng_bt = np.random.default_rng(20260811)  # own stream so pins survive cell re-ordering
Nyr_all = per_poc.groupby(["poc", per_poc.month.dt.year])["ICP count"].mean().unstack()
lev_n = {}
for y in range(2009, 2026):
    a = pd.read_parquet(ANA / f"{y}_pf_by_gxp.parquet",
                        columns=["trading_date", "trading_period", "gxp_code", "Q"])
    a = a[a.trading_period.isin(NIGHT)].copy()
    a["lead"] = -a.Q
    a["unit"] = a.gxp_code.map(site_of)
    a = a.dropna(subset=["unit"])
    tot = a.groupby(["unit", "trading_date", "trading_period"])["lead"].sum()
    lev_n[y] = tot.groupby("unit").median()
lev_n = pd.DataFrame(lev_n)
reg_of = {g: [g] for g in direct89}
for sid, (_, regs, _) in AGG.items():
    reg_of[sid] = regs
Nunit = pd.DataFrame({sid: Nyr_all.reindex(reg_of[sid]).sum(min_count=1) for sid in pan.index}).T
present_n = lev_n.dropna().index.intersection(Nunit.loc[:, range(2009, 2026)].dropna().index)
beta_rows = []
for y in range(2009, 2026):
    Xb = Nunit.loc[present_n, y].to_numpy(float)
    Yb = lev_n.loc[present_n, y].to_numpy(float)
    Ab = np.column_stack([np.ones(len(Xb)), Xb])
    bb, *_ = np.linalg.lstsq(Ab, Yb, rcond=None)
    bboots = [np.linalg.lstsq(Ab[i], Yb[i], rcond=None)[0][1] * 1e6
              for i in rng_bt.integers(0, len(Xb), (1000, len(Xb)))]
    lo_b, hi_b = np.percentile(bboots, [2.5, 97.5])
    beta_rows.append((y, len(Xb), bb[1] * 1e6, lo_b, hi_b))
bt = pd.DataFrame(beta_rows, columns=["year", "n", "beta", "lo", "hi"]).set_index("year")
print(bt.round(1).to_string())
d_beta = bt.beta.loc[2025] - bt.beta.loc[2013]
r1 = (bt.beta.loc[2013] - bt.beta.loc[2009]) / 4
r2_h = (bt.beta.loc[2019] - bt.beta.loc[2013]) / 6
r3 = (bt.beta.loc[2025] - bt.beta.loc[2019]) / 6
print(f"\nbeta(2025)-beta(2013) = {d_beta:.0f} VAr/ICP vs two-endpoint slope {slope_n:.0f}")
print(f"movement: 2009-13 {r1:.1f} (zero-crossing sweep) | 2013-19 {r2_h:.1f} | 2019-25 {r3:.1f} VAr/ICP/yr")
ok(len(present_n) == 92 and int(bt.n.min()) == 92,
   "beta(t) cohort: 92 of the 96 units present in every year 2009-2025")
ok(abs(bt.beta.loc[2009] + 40.7) < 2 and bt.hi.loc[2009] < 25,
   "2009 baseline: coefficient negative (-41 VAr/ICP; upper CI grazes zero at +5) - the residual motor-era inductive signature")
ok(bool((bt.beta.loc[[2009, 2010, 2011]] < 0).all()) and bool((bt.beta.loc[2012:] > 0).all()),
   "the coefficient crosses from inductive to capacitive between 2011 and 2012")
ok(bool((bt.beta.diff().loc[2012:] > 0).all()),
   "the coefficient rises every single year from 2012 onward - smooth, no steps")
ok(bool((bt.lo.loc[2016:] > 0).all()) and bt.lo.loc[2015] < 0, "interval excludes zero from 2016 onward (spans zero through 2015)")
ok(abs(bt.beta.loc[2013] - 50.1) < 2 and abs(bt.beta.loc[2019] - 141.4) < 2 and abs(bt.beta.loc[2025] - 262.4) < 2,
   "beta path pinned: 50 (2013) -> 141 (2019) -> 262 (2025) VAr/ICP")
ok(abs(d_beta - 212) < 3 and abs(d_beta - slope_n) < 0.1 * slope_n,
   "independent cross-check: beta(2025)-beta(2013) = 212 vs the two-endpoint slope 230 (within 10%)")
ok(abs(r2_h - 15.2) < 1 and abs(r3 - 20.2) < 1 and r3 > r2_h,
   "in-window accumulation accelerates: ~15 -> ~20 VAr/ICP/yr (2013-19 vs 2019-25)")

       n   beta     lo     hi
year                         
2009  92  -40.7  -73.7    4.8
2010  92  -53.1  -83.7    3.5
2011  92   -0.4  -65.5   76.8
2012  92   31.7  -41.0  113.7
2013  92   50.1  -24.9  136.2
2014  92   58.3  -12.7  145.4
2015  92   69.2   -6.7  137.1
2016  92   85.3   21.1  170.8
2017  92  106.3   42.2  180.1
2018  92  125.5   60.0  219.6
2019  92  141.4   81.4  214.9
2020  92  178.4  123.8  261.4
2021  92  203.7  157.6  284.6
2022  92  219.6  173.1  296.6
2023  92  222.3  184.9  284.9
2024  92  241.4  203.6  313.6
2025  92  262.4  222.7  322.6

beta(2025)-beta(2013) = 212 VAr/ICP vs two-endpoint slope 230
movement: 2009-13 22.7 (zero-crossing sweep) | 2013-19 15.2 | 2019-25 20.2 VAr/ICP/yr
  ok: beta(t) cohort: 92 of the 96 units present in every year 2009-2025
  ok: 2009 baseline: coefficient negative (-41 VAr/ICP; upper CI grazes zero at +5) - the residual motor-era inductive signature
  ok: the coefficient crosses from inductive to capacitive between 2011 and 201

In [13]:
# (b) The bottom-up order-of-magnitude device estimate - the hypothesis's testable
# prediction. Inputs are physics and datasheet-grade values, not NZ bench measurements:
# hypothesis-grade [H]. X-capacitor standing VAr at 230 V, 50 Hz: Q = omega * C * V^2.
OMEGA, V230 = 2 * np.pi * 50, 230.0
xcap = {uF: OMEGA * uF * 1e-6 * V230**2 for uF in (0.1, 0.47, 1.0)}
print("X-cap VAr at 230 V, 50 Hz:", {k: round(v, 1) for k, v in xcap.items()})
ok(abs(xcap[0.1] - 1.66) < 0.05 and abs(xcap[0.47] - 7.81) < 0.1 and abs(xcap[1.0] - 16.62) < 0.1,
   "X-capacitor standing VAr: 0.1 uF = 1.7, 0.47 uF = 7.8, 1.0 uF = 16.6")
res_lo, res_hi = HH_M * 20 * 2, HH_M * 40 * 4  # 20-40 always-connected devices x 2-4 VAr -> MVAr
com_lo, com_hi = 50.0, 300.0                   # commercial/industrial leg (same order; weakest data)
nat_lo, nat_hi = res_lo + com_lo, res_hi + com_hi
BAND = (150.0, 600.0)     # quoted national band (envelope tightened by judgment; central ~300)
PRED_HH = (100.0, 200.0)  # central per-household prediction band (<-> 200-400 MVAr national)
print(f"residential {res_lo:.0f}-{res_hi:.0f} MVAr + commercial {com_lo:.0f}-{com_hi:.0f}"
      f" = envelope {nat_lo:.0f}-{nat_hi:.0f}; quoted band {BAND[0]:.0f}-{BAND[1]:.0f}, central ~300")
ok((res_lo, res_hi) == (80.0, 320.0) and nat_lo <= BAND[0] and BAND[1] <= nat_hi,
   "residential 80-320 MVAr; national envelope (130-620) brackets the quoted 150-600 band")

X-cap VAr at 230 V, 50 Hz: {0.1: 1.7, 0.47: 7.8, 1.0: 16.6}
  ok: X-capacitor standing VAr: 0.1 uF = 1.7, 0.47 uF = 7.8, 1.0 uF = 16.6
residential 80-320 MVAr + commercial 50-300 = envelope 130-620; quoted band 150-600, central ~300
  ok: residential 80-320 MVAr; national envelope (130-620) brackets the quoted 150-600 band


In [14]:
# (b2) Tier-1 heat-pump slice of the residential estimate (added 7 Aug 2026, Dave's ask).
# Stock side [V/I]: heat pumps used in 66.8% of dwellings at the 2023 Census (47.3% in 2018)
#   - verified via EHINZ (Massey) indicator page reporting Stats NZ Census data, 7 Aug 2026;
#   units per using dwelling 1.2-1.6 [I assumption]; cross-check: EECA/Figure.NZ sales data
#   ~150k units/yr average since 2013 (record ~240k in 2023) -> ~1.9M sold in-window alone.
# Electrical side [H]: standing across-line filter capacitance 0.3-2.2 uF per unit
#   (datasheet-grade appliance EMI-filter values, NOT NZ-bench-measured - the bench
#   measurement stays the parked check).
PEN23, PEN18 = 0.668, 0.473
u_lo, u_hi = 1.2, 1.6
n_lo, n_hi = HH_M * PEN23 * u_lo, HH_M * PEN23 * u_hi        # millions of residential units
q_lo, q_hi = OMEGA * 0.3e-6 * V230**2, OMEGA * 2.2e-6 * V230**2   # VAr per unit
hp_lo, hp_hi = n_lo * q_lo, n_hi * q_hi                       # MVAr national residential
hp_central = HH_M * PEN23 * 1.4 * (OMEGA * 0.9e-6 * V230**2)
print(f"residential heat-pump units: {n_lo:.2f}-{n_hi:.2f}M "
      f"(census 66.8% x {u_lo}-{u_hi} units/dwelling; sales integral ~{0.150*13:.1f}M since 2013)")
print(f"per-unit standing: {q_lo:.1f}-{q_hi:.1f} VAr (0.3-2.2 uF at 230 V)")
print(f"heat-pump slice: {hp_lo:.0f}-{hp_hi:.0f} MVAr national residential (central ~{hp_central:.0f});"
      f" per household {hp_lo/HH_M:.0f}-{hp_hi/HH_M:.0f} VAr/hh")
ok(abs(q_lo - 5.0) < 0.1 and abs(q_hi - 36.6) < 0.3, "per-unit standing VAr band 5.0-36.6 (0.3-2.2 uF)")
ok(1.55 < n_lo < 1.65 and 2.1 < n_hi < 2.2 and abs(0.150 * 13 - 1.95) < 0.01,
   "units 1.60-2.14M; EECA-sales cross-check ~1.95M sold 2013-25 (same order)")
ok(7.9 <= hp_lo <= 9 and 77 <= hp_hi <= 80, "heat-pump slice ~8-78 MVAr, central ~28")
ok(hp_hi <= res_lo, "even the slice's UPPER bound sits at/below the residential envelope's FLOOR "
                    "(80 MVAr) - heat pumps are a material but minority class")
ok(hp_hi / HH_M < 113, "at most ~39 VAr/hh - the heat-pump fleet alone cannot carry the measured "
                       "113-161 VAr/hh rise; the swarm of smaller supplies does")

residential heat-pump units: 1.60-2.14M (census 66.8% x 1.2-1.6 units/dwelling; sales integral ~1.9M since 2013)
per-unit standing: 5.0-36.6 VAr (0.3-2.2 uF at 230 V)
heat-pump slice: 8-78 MVAr national residential (central ~28); per household 4-39 VAr/hh
  ok: per-unit standing VAr band 5.0-36.6 (0.3-2.2 uF)
  ok: units 1.60-2.14M; EECA-sales cross-check ~1.95M sold 2013-25 (same order)
  ok: heat-pump slice ~8-78 MVAr, central ~28
  ok: even the slice's UPPER bound sits at/below the residential envelope's FLOOR (80 MVAr) - heat pumps are a material but minority class
  ok: at most ~39 VAr/hh - the heat-pump fleet alone cannot carry the measured 113-161 VAr/hh rise; the swarm of smaller supplies does


In [15]:
# (c) The consistency comparison the paper states, plus the residual-sign check behind
# its residual sentence (industrial-mix buses under-rise relative to their connection
# count). Residuals from the REBASED panel (8 Aug).
ok(nat_lo < swing_p99 and swing < nat_hi and BAND[0] < swing_p99 and swing < BAND[1],
   "ORDER CHECK ONLY -- measured national rise (+227 to +292 MVAr) falls in the device estimate's "
   "numeric range, but the rise is a CHANGE and the estimate a standing LEVEL; see the "
   "'consistency' WARNING")
ok(PRED_HH[0] <= per_hh[0] and per_hh[-1] <= PRED_HH[1],
   "ORDER CHECK ONLY -- measured 113-161 VAr/hh falls in the predicted 100-200 VAr/hh central "
   "band numerically, but measured is a per-household CHANGE and predicted a standing LEVEL; "
   "see the 'consistency' WARNING")
ok(float(pan.loc["TAK0331", "resid"]) < 0 and float(pan.loc["WIR0331", "resid"]) < 0,
   "industrial-mix Takanini and Wiri sit below the fitted line (under-rise)")
ok(float(pan.loc["PEN0221+0331", "resid"]) > 0,
   "the Penrose group sits far above the line (over-rise: CBD cable network rides on the fleet)")
print(f"consistency: measured rise {swing_p99:+.0f}..{swing:+.0f} MVAr "
      f"({per_hh[0]:.0f}-{per_hh[-1]:.0f} VAr/hh)  vs  bottom-up {BAND[0]:.0f}-{BAND[1]:.0f} MVAr "
      f"({PRED_HH[0]:.0f}-{PRED_HH[1]:.0f} VAr/hh central)")
print(f"residuals (rebased): TAK {pan.loc['TAK0331','resid']:+.1f}, WIR {pan.loc['WIR0331','resid']:+.1f}, "
      f"PEN group {pan.loc['PEN0221+0331','resid']:+.1f} MVAr")

  ok: ORDER CHECK ONLY -- measured national rise (+227 to +292 MVAr) falls in the device estimate's numeric range, but the rise is a CHANGE and the estimate a standing LEVEL; see the 'consistency' WARNING
  ok: ORDER CHECK ONLY -- measured 113-161 VAr/hh falls in the predicted 100-200 VAr/hh central band numerically, but measured is a per-household CHANGE and predicted a standing LEVEL; see the 'consistency' WARNING
  ok: industrial-mix Takanini and Wiri sit below the fitted line (under-rise)
  ok: the Penrose group sits far above the line (over-rise: CBD cable network rides on the fleet)
consistency: measured rise +227..+292 MVAr (113-161 VAr/hh)  vs  bottom-up 150-600 MVAr (100-200 VAr/hh central)
residuals (rebased): TAK -5.8, WIR -5.1, PEN group +3.7 MVAr


In [16]:
# Notebook figure, TWO PANELS (7 Aug; REBASED 8 Aug to the 96-unit panel): A = the
# two-endpoint scatter; B = the coefficient's history beta(t). Paper sign convention
# (Q < 0 = leading) throughout, so changes and coefficients plot negative.
#
# NOT THE PAPER'S FIGURE (corrected 24 Aug 2026, Pass E). Fig. 5 in both tiers is built by
# ../MAKE_FIG_CONNECTION_SCALING.py, which re-derives this notebook's state (it executes the
# cells above and stops at THIS one), then draws the exhibit at column width, serif, 400 dpi,
# and emits both .png and the .pdf main.tex actually includes. This cell used to save over
# that file at 130 dpi from a 7.2 x 9.0 in canvas scaled to 0.94 columnwidth -- every stroke
# divided by 2.2 -- and its panel-B title asserted a crossing DATE the paper has since
# retracted. It now writes beside the notebook instead, and carries the corrected title.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGDIR = HERE          # the notebook's own directory -- see the header note
EA_BLUE, GREEN = "#003366", "#2ca02c"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 130, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": EA_BLUE, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.facecolor": "white",
})
fig, (axA, axB) = plt.subplots(2, 1, figsize=(7.2, 9.0))
axA.axhline(0, color="0.6", lw=0.8, ls="--", zorder=1)
axA.scatter(Xn / 1000, -Yn, s=26, c=EA_BLUE, alpha=0.75, lw=0, zorder=3)
xx = np.linspace(0, Xn.max() * 1.04, 50)
axA.plot(xx / 1000, -(b1n[0] + b1n[1] * xx), c=GREEN, lw=1.8, zorder=2)
for code, label, dxy in [("PEN0221+0331", "Penrose", (5, 4)), ("ALB+WRD", "North Shore", (5, 4)),
                         ("HEN+HEP", "West Auckland", (5, 4)), ("HWB+SDN", "Dunedin", (5, 4)),
                         ("TAK0331", "TAK0331", (2, -12))]:
    r = pan.loc[code]
    axA.annotate(label, (r.Nbar / 1000, -r.dQ), textcoords="offset points",
                 xytext=dxy, fontsize=7, color="0.35")
axA.set_xlabel("Connections served (mean of 2013 and 2025 ICP count, thousands)")
axA.set_ylabel("Change in median overnight reactive power\n2013-2025 (MVAr)")
axA.set_title("A. The overnight deepening scales with connections served")
_iq = -b1n[0]  # Q-signed intercept, as the drawn line has it
axA.annotate(f"slope $-${abs(slope_n):.0f} VAr per connection\n"
             f"bootstrap 95% CI $-${abs(ci_n[1]):.0f} to $-${abs(ci_n[0]):.0f}\n"
             f"intercept {'$-$' if _iq < 0 else '$+$'}{abs(_iq):.2f} MVAr per unit,  $R^2$ = {r2n:.2f},  n = {npan}",
             xy=(0.97, 0.95), xycoords="axes fraction", ha="right", va="top", fontsize=10)
axB.axhline(0, color="0.6", lw=0.8, ls="--", zorder=1)
axB.fill_between(bt.index, -bt.hi, -bt.lo, color=EA_BLUE, alpha=0.15, lw=0, zorder=2)
axB.plot(bt.index, -bt.beta, "o-", c=EA_BLUE, lw=1.8, ms=4, zorder=3)
axB.set_xlabel("Year")
axB.set_ylabel("Standing VAr per connection\n(cross-sectional coefficient)")
axB.set_title("B. The coefficient's history: inductive early in the record,\ndeepening capacitive every year since")
axB.annotate(f"accumulation rate\n2013-19 $\\approx$ {r2_h:.0f}, 2019-25 $\\approx$ {r3:.0f}\nVAr per connection per year",
             xy=(0.03, 0.06), xycoords="axes fraction", va="bottom", fontsize=10)
fig.tight_layout()
fig.savefig(FIGDIR / "06_connection_scaling_notebook.png")
print(f"notebook figure (2 panels, rebased n={npan}) -> "
      f"{FIGDIR / '06_connection_scaling_notebook.png'}  "
      "[the paper's Fig. 5 is built by ../MAKE_FIG_CONNECTION_SCALING.py]")
plt.show()

notebook figure (2 panels, rebased n=96) -> /home/dave/gridlytics/clients/ea/power-factor/ieee-pf-trajectory-paper/icp-regression/06_connection_scaling_notebook.png  [the paper's Fig. 5 is built by ../MAKE_FIG_CONNECTION_SCALING.py]


In [17]:
res = {
 "panel_rebase_20260808": {
   "cohort_n": 114, "panel_units": npan, "members_represented": 103,
   "excluded_industrial": sorted(EXCL_A), "excluded_no_endpoint": sorted(EXCL_B),
   "excluded_unresolvable": sorted(EXCL_E),
   "aggregation_units": {sid: mem for sid, (mem, _, _) in AGG.items()},
   "note": "correspondence audit 8 Aug 2026: registry root-NSP attribution vs metering GXP "
           "boundaries; churn groups evidence-classed by the migration screen (P-step matched)"},
 "cluster_inference_20260810": {
   "note": "peer-review response (9 Aug panel): company as the resampling unit. Company = 2025 "
           "canonical POC->nwk mapping where it names an EDB, else the registry root NSP "
           "(PEN0331 maps to SHPK, absent from the registry; TKU0331 to a generator code).",
   "n_companies": int(G), "units_per_company_mean": round(npan / G, 1),
   "largest_by_units": [str(sz.index[0]), int(sz.iloc[0])],
   "largest_by_connections": [str(shareN.index[0]), round(float(shareN.iloc[0]), 3)],
   "slope_var_per_icp": {
     "point": round(slope_n, 1),
     "iid_pairs_boot": [round(float(c), 1) for c in ci_n],
     "company_cluster_boot": [round(float(c), 1) for c in cl_sl_ci],
     "classical_ols": [round(slope_n - tN * float(np.sqrt(Vcl[1, 1])) * V, 1),
                       round(slope_n + tN * float(np.sqrt(Vcl[1, 1])) * V, 1)],
     "hc1": [round(slope_n - tN * float(np.sqrt(Vhc[1, 1])) * V, 1),
             round(slope_n + tN * float(np.sqrt(Vhc[1, 1])) * V, 1)],
     "cr1_cluster_t": [round(slope_n - tG * float(np.sqrt(Vcr[1, 1])) * V, 1),
                       round(slope_n + tG * float(np.sqrt(Vcr[1, 1])) * V, 1)],
     "wild_cluster_boot_p": round(p_sl, 4)},
   "intercept_mvar_per_unit": {
     "point": round(float(b1n[0]), 2),
     "aggregate_mvar": round(float(b1n[0]) * npan, 0),
     "share_of_369mvar_deepening": round(abs(float(b1n[0]) * npan) / 369, 3),
     "iid_pairs_boot": [round(float(c), 2) for c in ici_n],
     "company_cluster_boot": [round(float(c), 2) for c in cl_in_ci],
     "classical_ols": [round(float(b1n[0] - tN * np.sqrt(Vcl[0, 0])), 2),
                       round(float(b1n[0] + tN * np.sqrt(Vcl[0, 0])), 2)],
     "hc1": [round(float(b1n[0] - tN * np.sqrt(Vhc[0, 0])), 2),
             round(float(b1n[0] + tN * np.sqrt(Vhc[0, 0])), 2)],
     "cr1_cluster_t": [round(float(b1n[0] - tG * np.sqrt(Vcr[0, 0])), 2),
                       round(float(b1n[0] + tG * np.sqrt(Vcr[0, 0])), 2)],
     "wild_cluster_boot_p": round(p_in, 4),
     "reading": "ALL SCHEMES SPAN ZERO: i.i.d., classical, HC1 and both company-clustered "
                "schemes agree; wild cluster bootstrap-t p = %s. No connection-independent term "
                "is detectable - the device story's zero-intercept prediction is met." % round(p_in, 4)},
   "cr1_caveat": "CR1 SEs fall BELOW HC1 SEs (slope %s vs %s) at G=%s with one %s-unit cluster; "
                 "CR estimators are least reliable in this regime and must not be used to tighten "
                 "a claim. The wild cluster bootstrap is the reported test." % (
                     round(float(np.sqrt(Vcr[1, 1])) * V, 1), round(float(np.sqrt(Vhc[1, 1])) * V, 1),
                     int(G), int(sz.iloc[0])),
   "leave_one_company_out_var_per_icp": [round(float(loco_s.min()), 1), round(float(loco_s.max()), 1)],
   "leave_one_company_out_detail": {str(c): round(float(v), 1) for c, v in loco.items()},
   "scale_free": {"per_connection_mean": round(float(Yp.mean()), 0),
                  "per_connection_median": round(float(np.median(Yp)), 0),
                  "rate_on_size_r2": round(float(r2sf), 3),
                  "note": "per-connection rate is size-invariant (R2 ~ 0), so the extensive R2=0.78 "
                          "is not merely a size artefact; unweighted site mean %s vs size-weighted OLS %s"
                          % (round(float(Yp.mean())), round(slope_n))}},
 "M0_null": {"aic": round(aicn0, 1)},
 "M1": {"slope_var_per_icp": round(slope_n, 1),
        "ci95_var_per_icp": [round(float(c), 1) for c in ci_n],
        "intercept_mvar": round(float(b1n[0]), 2),
        "intercept_ci95_mvar": [round(float(c), 2) for c in ici_n],
        "r2": round(r2n, 3), "aic": round(aicn1, 1)},
 "M1_theilsen_var_per_icp": round(ts_n, 1),
 "M1_through_origin_var_per_icp": round(origin_n, 1),
 "M2": {"a_var_per_icp": round(b2n[1] * V, 1), "b_var_per_new_icp": round(b2n[2] * V, 1),
        "b_ci95": [round(float(c), 1) for c in m2b_ci],
        "r2": round(r2n_2, 3), "aic": round(aicn2, 1), "corr_Nbar_dN": round(corr_n, 2),
        "note": "split unidentified: b's CI spans zero at collinearity 0.78; M2 adds no AIC"},
 "loo_slope_range_var_per_icp": [round(float(loo_n.min()), 1), round(float(loo_n.max()), 1)],
 "prediction_band_var_per_icp": [50, 400],
 "verdict": "SUPPORTED (pre-specified 6-Aug rule, applied unchanged to the rebased panel): "
            "slope in band, CI excludes 0, R2 >> 0.15",
 "beta_t_history": {
     "n": int(len(present_n)), "years": [2009, 2025],
     "beta_var_per_icp": {str(y): round(float(bt.beta.loc[y]), 1) for y in bt.index},
     "endpoint_diff_var_per_icp": round(float(d_beta), 1),
     "movement_var_per_icp_yr": {"2009-13_zero_crossing": round(float(r1), 1),
                                 "2013-19": round(float(r2_h), 1), "2019-25": round(float(r3), 1)},
     "notes": "cross-sectional level slope per year, same 92 units; negative 2009-2011 "
              "(inductive fleet; 2009 upper CI grazes zero), crosses zero 2012, monotone rise from "
              "2012, CI excludes 0 from 2016; endpoint diff %d agrees with the two-endpoint slope "
              "%d within 10%%" % (round(d_beta), round(slope_n))},
 "legacy_panel_20260806": {
     "note": "original pre-registered run on the fault-level-constrained screen-table panel; "
             "kept verbatim as the labelled consistency result - report both, never blend",
     "n_clean_joined": n, "n_all_joined": int(len(d_all)),
     "M1": {"slope_var_per_icp": round(slope, 1), "ci95_var_per_icp": [round(float(c), 1) for c in ci],
            "intercept_mvar": round(float(b1[0]), 2), "r2": round(r2, 3)},
     "M1_theilsen_var_per_icp": round(ts, 1), "M1_through_origin_var_per_icp": round(origin, 1),
     "M2": {"a_var_per_icp": round(b2[1] * V, 1), "b_var_per_new_icp": round(b2[2] * V, 1),
            "r2": round(r2_2, 3), "corr_Nbar_dN": round(corr, 2)},
     "loo_slope_range_var_per_icp": [round(float(loo.min()), 1), round(float(loo.max()), 1)],
     "beta_t_2013_2019_2025": [27.3, 131.1, 273.8],
     "verdict": "SUPPORTED (original run of the same rule, 6 Aug 2026)"},
 "national_sums_clean90": {
     "n": int(len(cb)), "night_med_2013_mvar": round(s13, 1), "night_med_2025_mvar": round(s25, 1),
     "swing_mvar": round(swing, 1), "risers_mvar": round(risers, 1), "fallers_mvar": round(fallers, 1),
     "swing_p99_mvar": round(swing_p99, 1),
     "per_household_var": [round(per_hh[0], 1), round(per_hh[-1], 1)], "households_m": HH_M,
     "note": "fault-level-constrained screen-table subset; the legacy regression panel derives "
             "from this; the PAPER quotes the 114-site archive-native sums below"},
 "national_sums_clean114": {
     "n": n114, "night_med_2013_mvar": round(s13_114, 1), "night_med_2025_mvar": round(s25_114, 1),
     "swing_mvar": round(swing114, 1), "risers_mvar": round(risers114, 1),
     "fallers_mvar": round(fallers114, 1), "swing_p99_mvar": round(swing114_p99, 1),
     "per_household_var": [round(per_hh114[0], 1), round(per_hh114[-1], 1)], "households_m": HH_M,
     "parallel_shift_native": {"night_rise_med_mvar": round(float(dq114.median()), 2),
                               "peak_rise_med_mvar": round(float(dk114.median()), 2),
                               "ratio_med": round(ratio114, 2),
                               "peak_load_chg_pct": round(pload114, 1)}},
 "device_estimate": {
     "xcap_var_at_230v": {str(k): round(v, 2) for k, v in xcap.items()},
     "residential_mvar": [res_lo, res_hi], "commercial_mvar": [com_lo, com_hi],
     "national_envelope_mvar": [nat_lo, nat_hi], "quoted_band_mvar": list(BAND),
     "per_household_central_var": list(PRED_HH)},
 "heatpump_slice_tier1": {
     "census_penetration": {"2018": PEN18, "2023": PEN23},
     "residential_units_m": [round(n_lo, 2), round(n_hi, 2)],
     "per_unit_var": [round(q_lo, 1), round(q_hi, 1)], "per_unit_uF": [0.3, 2.2],
     "slice_mvar": [round(hp_lo, 1), round(hp_hi, 1)], "central_mvar": round(hp_central, 0),
     "per_household_var_max": round(hp_hi / HH_M, 1),
     "published_band_note": "SUPERSEDED FOR PUBLICATION -- both paper tiers print the LATER reference-design band: "
                         "1.4-2.0 uF, 23-33 VAr per unit, 37-69 MVAr for the class (extended Appendix B, Table "
                         "IV, heat-pump / AC outdoor-unit row; journal Section V-B). That band resolved the "
                         "outdoor unit specifically in the device-fleet reference-design review "
                         "(research/device-fleet/, 9 Aug 2026). The 0.3-2.2 uF here is the wider 7 Aug tier-1 "
                         "appliance-filter bracket, kept verbatim as the dated first estimate -- do not report "
                         "it as the paper's band. Both agree on what the slice is for: material but a minority "
                         "of the 80-320 MVAr residential envelope.",
     "verdict": "material but minority: upper bound ~ residential envelope floor; cannot carry the "
                "measured per-household rise (113-161 on the 90-panel; 157-214 on the 114-panel) - "
                "the small-supply swarm does"},
 "consistency": "measured 2013-25 deepening (114-site panel: +314 to +369 MVAr; 157-214 VAr/hh). WARNING "
                "(external review 2, 24 Aug 2026): this is a twelve-year CHANGE; the bottom-up 150-600 "
                "MVAr / 100-200 VAr-per-household band is a standing LEVEL. Do NOT report the change as "
                "being inside the band -- that comparison was inherited by figures/07_class_split.tex and "
                "printed in both tiers before it was caught. The level comparator is the matched-vintage "
                "residential level, -310 VAr/connection in 2025, which lands ABOVE the band because it "
                "carries each connection's share of MV cable. Paper Section V-C.",
 "paper_figure": "replication/figures/06_connection_scaling.pdf (2 panels: scatter + beta_t, "
                 "rebased n=96). BUILT BY ../MAKE_FIG_CONNECTION_SCALING.py, not by this "
                 "notebook's own figure cell: that script re-derives the state above and draws "
                 "the exhibit at column width / serif / 400 dpi, emitting the .pdf both tiers "
                 "include plus a matching .png.",
}
(HERE / "icp_regression_results_20260806.json").write_text(json.dumps(res, indent=1))
print(json.dumps(res["panel_rebase_20260808"], indent=1))
print(json.dumps(res["M1"], indent=1))
print(f"\nASSERTIONS PASSED: {PASSED}/{PASSED}")

{
 "cohort_n": 114,
 "panel_units": 96,
 "members_represented": 103,
 "excluded_industrial": [
  "BDE0111",
  "GLN0331",
  "KIN0111",
  "KIN0113",
  "LFD1101",
  "LFD1102",
  "TWI2201",
  "WHI0111"
 ],
 "excluded_no_endpoint": [
  "ASY0111",
  "MNG1101"
 ],
 "excluded_unresolvable": [
  "LTN0331"
 ],
 "aggregation_units": {
  "KBY0661+2": [
   "KBY0661",
   "KBY0662"
  ],
  "HTI0331+1101": [
   "HTI0331",
   "HTI1101"
  ],
  "ALB+WRD": [
   "ALB0331",
   "ALB1101",
   "WRD0331"
  ],
  "HEN+HEP": [
   "HEN0331",
   "HEP0331"
  ],
  "PEN0221+0331": [
   "PEN0221",
   "PEN0331"
  ],
  "CPK0111+0331": [
   "CPK0111",
   "CPK0331"
  ],
  "HWB+SDN": [
   "HWB0331",
   "SDN0331"
  ]
 },
 "note": "correspondence audit 8 Aug 2026: registry root-NSP attribution vs metering GXP boundaries; churn groups evidence-classed by the migration screen (P-step matched)"
}
{
 "slope_var_per_icp": 229.6,
 "ci95_var_per_icp": [
  198.6,
  272.2
 ],
 "intercept_mvar": 0.29,
 "intercept_ci95_mvar": [
  -0.21,
 

## Verdict and caveats

**SUPPORTED under the pre-specified rule, on both panels.** On the rebased 96-unit panel the twelve-year rise scales linearly with connections at ~230 VAr per ICP (95% CI 199-272), intercept indistinguishable from zero under every inference scheme (+0.3 MVAr; all five CIs span 0, wild cluster bootstrap-t p = 0.29 - the r3 cell), R^2 = 0.80; Theil-Sen 259 and through-origin 237 agree; leave-one-out spans 220-242; the bus-level-artefact null loses by dAIC ~ 155. The original pre-registered 83-site run (kept verbatim above) reads ~253 (CI 192-291, R^2 = 0.77); it was never exposed to the duplicate-row defect (per-site screen-table medians, no aggregation groups - see History, 13 Aug), and the corrected rebased panel agrees with it within ~10%. Report both, never blend.

**The coefficient's history is the paper's arc in one number.** On the 92 units present throughout: **-41 VAr/ICP in 2009** (point estimate negative - the residual motor-era inductive signature; the upper CI grazes zero), crossing zero at 2012, then rising every single year to **+262 in 2025** (CI clear of zero from 2016). In-window accumulation accelerates ~15 (2013-19) to ~20 (2019-25) VAr/ICP/yr. The endpoint difference (212) independently reproduces the two-endpoint slope (230) within 10%.

**Caveats, in force.** (1) Across buses, ICP count is collinear with bus scale - strictly this demonstrates clean linear scaling with connections served; no plausible metering mechanism produces that with zero intercept, so the metering-registration rival is heavily disfavoured, not formally dead. (2) The slope is the whole connection-scaling class: device filter capacitance PLUS each connection's share of MV cable (LV cable is negligible at 400 V - Q scales with V^2). The split within the slope belongs to the LV distribution-transformer night measurement. (3) Sitting at the top of the 100-200 VAr/household central device estimate is expected: commercial ICPs carry larger filter fleets and the cable share rides along. (4) Industrial-mix buses (Takanini, Wiri) under-rise - sensible scatter, no pathology. (5) The M2 growth split is unidentified (collinearity 0.78, b's CI spans zero, AIC-indistinguishable from M1) - the accumulation-vs-new-connection decomposition needs the beta(t) route, not M2. (6) Bus-level registry attribution is nominal: the correspondence audit aggregates the demonstrated churn clusters, and sub-threshold churn (< ~5% / 2,000 ICPs at the endpoints) remains as noise the robustness estimators absorb; the Penrose group carries a documented ~7% under-count leak to its contaminated 110 kV sibling (dropping the group entirely: slope 220, verdict unchanged).

**Next:** LV distribution-transformer night measurement (isolates the device fleet: LV-side metering sees no transformer magnetising and no meaningful cable charging); EDB LV-monitor-fleet data request is the force-multiplied version. Paper use: Sections V-B / V-C of `../main.tex` and the extended version; FSR 2026 scaling sentence; harmonics W6.

**History.** 6 Aug 2026: pre-registered rule fixed, first run (83-site screen-table panel), SUPPORTED. 7 Aug: national sums (n = 90), device estimate, heat-pump slice, consistency comparison, paper figure; then D1 archive-native 114-site sums (paper V-B numbers) and the native parallel-shift cross-check; beta(t) history added (81 sites). **8 Aug: panel rebase** - correspondence audit (industrial identities, registry re-registrations, churn groups with the migration screen and P-step classification), headline moved to the 96-unit archive-native panel, beta(t) rebased to 92 units; legacy run retained verbatim; battery extended. The screen-table agreement check ties the two panels' measurement paths together. **10 Aug: cluster-robust inference (r3)** - company as the resampling unit, peer-review response. **13 Aug: archive correction (pass-7 R3).** The shipped `data/analysis` vintage predated the 14-Mar-2026 processed rebuild and carried duplicate rows at dual-network POCs (the full POC Q repeated on each row), which the half-hourly member sums of the group construction double-counted at HEP0331 (HEN+HEP, all years), PEN0331 (Penrose, all years) and HWB0331 (HWB+SDN, 2015 on). Archive rebuilt from the current processed files: slope 316 -> 230, R^2 0.78 -> 0.80, the 10-Aug scheme-dependent intercept detectability disappears (it was the duplicates - the intercept now spans zero under every scheme), beta path 54/166/345 -> 50/141/262, peak-load print +3.4% -> +2.9% (the P_net definition). Per-site medians - the national sums, the deepening, the parallel-shift test - are unchanged. The legacy panel consumes the screen table (per-site medians, dedup-immune) and is untouched.